# DhakaNest — Shared Data Preparation

This notebook prepares the common dataset and fixed data splits used by
Random Forest, XGBoost, and CatBoost.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

Mounted at /content/drive


In [ ]:
from pathlib import Path

MY_DRIVE = Path("/content/drive/MyDrive")

print("My Drive exists:", MY_DRIVE.exists())

My Drive exists: True


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/DhakaNest_ML")

DIRECTORIES = [
    PROJECT_ROOT / "data" / "raw",
    PROJECT_ROOT / "data" / "interim",
    PROJECT_ROOT / "data" / "processed",
    PROJECT_ROOT / "data" / "splits",

    PROJECT_ROOT / "notebooks",

    PROJECT_ROOT / "reports" / "dataset_audit",
    PROJECT_ROOT / "reports" / "figures" / "random_forest",
    PROJECT_ROOT / "reports" / "figures" / "xgboost",
    PROJECT_ROOT / "reports" / "figures" / "catboost",
    PROJECT_ROOT / "reports" / "figures" / "comparison",
    PROJECT_ROOT / "reports" / "metrics",
    PROJECT_ROOT / "reports" / "segment_evaluation",

    PROJECT_ROOT / "artifacts" / "candidates" / "random_forest",
    PROJECT_ROOT / "artifacts" / "candidates" / "xgboost",
    PROJECT_ROOT / "artifacts" / "candidates" / "catboost",
    PROJECT_ROOT / "artifacts" / "fallback_models",
    PROJECT_ROOT / "artifacts" / "winning_model_bundle",

    PROJECT_ROOT / "manifests",
    PROJECT_ROOT / "logs",
    PROJECT_ROOT / "exports",
]

for directory in DIRECTORIES:
    directory.mkdir(parents=True, exist_ok=True)

print(f"Created or verified {len(DIRECTORIES)} directories.")
print("Project root:", PROJECT_ROOT)

Created or verified 20 directories.
Project root: /content/drive/MyDrive/DhakaNest_ML


In [ ]:
for path in sorted(PROJECT_ROOT.rglob("*")):
    if path.is_dir():
        print(path.relative_to(PROJECT_ROOT))

artifacts
artifacts/candidates
artifacts/candidates/catboost
artifacts/candidates/random_forest
artifacts/candidates/xgboost
artifacts/fallback_models
artifacts/winning_model_bundle
data
data/interim
data/processed
data/raw
data/splits
exports
logs
manifests
notebooks
reports
reports/dataset_audit
reports/figures
reports/figures/catboost
reports/figures/comparison
reports/figures/random_forest
reports/figures/xgboost
reports/metrics
reports/segment_evaluation


In [ ]:
from pathlib import Path

PROJECT_ROOT = Path("/content/drive/MyDrive/DhakaNest_ML")

RAW_DIR = PROJECT_ROOT / "data" / "raw"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
SPLITS_DIR = PROJECT_ROOT / "data" / "splits"

REPORTS_DIR = PROJECT_ROOT / "reports"
FIGURES_DIR = REPORTS_DIR / "figures"
METRICS_DIR = REPORTS_DIR / "metrics"
SEGMENT_EVALUATION_DIR = REPORTS_DIR / "segment_evaluation"

ARTIFACTS_DIR = PROJECT_ROOT / "artifacts"
CANDIDATES_DIR = ARTIFACTS_DIR / "candidates"
FALLBACK_MODELS_DIR = ARTIFACTS_DIR / "fallback_models"
WINNER_DIR = ARTIFACTS_DIR / "winning_model_bundle"

MANIFESTS_DIR = PROJECT_ROOT / "manifests"
LOGS_DIR = PROJECT_ROOT / "logs"
EXPORTS_DIR = PROJECT_ROOT / "exports"

RAW_DATA_PATH = RAW_DIR / "houserentdhaka.csv"
PROCESSED_DATA_PATH = PROCESSED_DIR / "dhakanest_rent_training_v1.csv"

TRAIN_PATH = SPLITS_DIR / "train.csv"
VALIDATION_PATH = SPLITS_DIR / "validation.csv"
TEST_PATH = SPLITS_DIR / "test.csv"
SPLIT_MANIFEST_PATH = SPLITS_DIR / "split_manifest.csv"

print("Project paths configured.")

Project paths configured.


In [ ]:
import pandas as pd

assert RAW_DATA_PATH.exists(), (
    f"Dataset was not found at:\n{RAW_DATA_PATH}"
)

raw_df = pd.read_csv(RAW_DATA_PATH)

print("Dataset shape:", raw_df.shape)
print("Dataset columns:", raw_df.columns.tolist())
raw_df.head()

Dataset shape: (28800, 6)
Dataset columns: ['Unnamed: 0', 'Location', 'Area', 'Bed', 'Bath', 'Price']


,Unnamed: 0,Location,Area,Bed,Bath,Price
0,0,"Block H, Bashundhara R-A, Dhaka","1,600 sqft",3,3,20 Thousand
1,1,"Farmgate, Tejgaon, Dhaka",900 sqft,2,2,20 Thousand
2,2,"Block B, Nobodoy Housing Society, Mohammadpur,...","1,250 sqft",3,3,18 Thousand
3,3,"Gulshan 1, Gulshan, Dhaka","2,200 sqft",3,4,75 Thousand
4,4,"Baridhara, Dhaka","2,200 sqft",3,3,75 Thousand


In [ ]:
EXPECTED_SHAPE = (28800, 6)

EXPECTED_COLUMNS = [
    "Unnamed: 0",
    "Location",
    "Area",
    "Bed",
    "Bath",
    "Price",
]

assert raw_df.shape == EXPECTED_SHAPE, (
    f"Unexpected dataset shape: {raw_df.shape}"
)

assert raw_df.columns.tolist() == EXPECTED_COLUMNS, (
    f"Unexpected columns: {raw_df.columns.tolist()}"
)

print("Raw dataset validation passed.")

Raw dataset validation passed.


In [ ]:
import hashlib

def calculate_sha256(file_path: Path) -> str:
    sha256 = hashlib.sha256()

    with file_path.open("rb") as file:
        for block in iter(lambda: file.read(1024 * 1024), b""):
            sha256.update(block)

    return sha256.hexdigest()


raw_dataset_sha256 = calculate_sha256(RAW_DATA_PATH)

print("Dataset SHA-256:")
print(raw_dataset_sha256)

Dataset SHA-256:
bcd4590a3520d072e8e9a091b832c03a110cbbe1a877bd5b0fab9f8a09f0078b


In [ ]:
import json
from datetime import datetime, timezone

raw_dataset_manifest = {
    "filename": RAW_DATA_PATH.name,
    "rows": int(raw_df.shape[0]),
    "columns": int(raw_df.shape[1]),
    "column_names": raw_df.columns.tolist(),
    "sha256": raw_dataset_sha256,
    "verified_at_utc": datetime.now(timezone.utc).isoformat(),
}

manifest_path = MANIFESTS_DIR / "raw_dataset_manifest.json"

with manifest_path.open("w", encoding="utf-8") as file:
    json.dump(raw_dataset_manifest, file, indent=2)

print("Saved:", manifest_path)

Saved: /content/drive/MyDrive/DhakaNest_ML/manifests/raw_dataset_manifest.json


In [10]:
project_readme = """# DhakaNest ML Workspace

This directory contains the complete machine-learning workflow for the
DhakaNest rent-prediction system.

## Dataset

The original dataset is stored at:

data/raw/houserentdhaka.csv

The raw dataset must never be edited manually.

## Notebooks

- 00_DhakaNest_Data_Preparation.ipynb
- 01_DhakaNest_Random_Forest.ipynb
- 02_DhakaNest_XGBoost.ipynb
- 03_DhakaNest_CatBoost.ipynb
- 04_DhakaNest_Model_Comparison_and_Export.ipynb

## Evaluation Metrics

Rent-prediction candidates are evaluated using:

- MAE
- RMSE
- R²
- MAPE
- Median Absolute Error
- P90 Absolute Error

## Fixed Configuration

- Random state: 42
- Training split: 70%
- Validation split: 15%
- Test split: 15%
"""

readme_path = PROJECT_ROOT / "README.md"

readme_path.write_text(project_readme, encoding="utf-8")

print("Saved:", readme_path)

Saved: /content/drive/MyDrive/DhakaNest_ML/README.md


In [11]:
project_config = {
    "project_name": "DhakaNest",
    "model_domain": "monthly_rent_prediction",
    "currency": "BDT",
    "random_state": 42,
    "train_ratio": 0.70,
    "validation_ratio": 0.15,
    "test_ratio": 0.15,
    "raw_dataset": "houserentdhaka.csv",
    "feature_columns": [
        "broad_area",
        "model_micro_area",
        "area_sqft",
        "bedrooms",
        "bathrooms",
    ],
    "target_column": "base_rent_bdt",
    "candidate_models": [
        "random_forest",
        "xgboost",
        "catboost",
    ],
    "regression_metrics": [
        "MAE",
        "RMSE",
        "R2",
        "MAPE",
        "Median Absolute Error",
        "P90 Absolute Error",
    ],
}

config_path = MANIFESTS_DIR / "project_config.json"

with config_path.open("w", encoding="utf-8") as file:
    json.dump(project_config, file, indent=2)

print("Saved:", config_path)

Saved: /content/drive/MyDrive/DhakaNest_ML/manifests/project_config.json


In [12]:
from pathlib import Path
import pandas as pd
import json

required_directories = [
    RAW_DIR,
    INTERIM_DIR,
    PROCESSED_DIR,
    SPLITS_DIR,
    REPORTS_DIR,
    FIGURES_DIR,
    METRICS_DIR,
    ARTIFACTS_DIR,
    CANDIDATES_DIR,
    MANIFESTS_DIR,
    EXPORTS_DIR,
]

missing_directories = [
    str(path)
    for path in required_directories
    if not path.exists()
]

if missing_directories:
    raise FileNotFoundError(
        "Missing directories:\n" + "\n".join(missing_directories)
    )

if not RAW_DATA_PATH.exists():
    raise FileNotFoundError(
        f"Raw dataset missing: {RAW_DATA_PATH}"
    )

verification_df = pd.read_csv(RAW_DATA_PATH)

if verification_df.shape != (28800, 6):
    raise ValueError(
        f"Unexpected dataset shape: {verification_df.shape}"
    )

expected_columns = [
    "Unnamed: 0",
    "Location",
    "Area",
    "Bed",
    "Bath",
    "Price",
]

if verification_df.columns.tolist() != expected_columns:
    raise ValueError(
        f"Unexpected dataset columns: "
        f"{verification_df.columns.tolist()}"
    )

required_files = [
    PROJECT_ROOT / "README.md",
    MANIFESTS_DIR / "project_config.json",
    MANIFESTS_DIR / "raw_dataset_manifest.json",
]

missing_files = [
    str(path)
    for path in required_files
    if not path.exists()
]

if missing_files:
    raise FileNotFoundError(
        "Missing setup files:\n" + "\n".join(missing_files)
    )

print("=" * 60)
print("DHAKANEST GOOGLE DRIVE SETUP PASSED")
print("=" * 60)
print("Project root:", PROJECT_ROOT)
print("Dataset shape:", verification_df.shape)
print("Dataset columns:", verification_df.columns.tolist())
print("The workspace is ready for data preparation.")

DHAKANEST GOOGLE DRIVE SETUP PASSED
Project root: /content/drive/MyDrive/DhakaNest_ML
Dataset shape: (28800, 6)
Dataset columns: ['Unnamed: 0', 'Location', 'Area', 'Bed', 'Bath', 'Price']
The workspace is ready for data preparation.


# Part 2 — Dataset Preparation

This section audits, cleans, normalizes, validates, and freezes the common
dataset used by Random Forest, XGBoost, and CatBoost.

In [13]:
!pip install -q pandas numpy scikit-learn matplotlib joblib

In [14]:
from pathlib import Path
from datetime import datetime, timezone
from io import StringIO

import hashlib
import json
import platform
import re
import unicodedata
import warnings

import numpy as np
import pandas as pd
import sklearn

from sklearn.model_selection import GroupShuffleSplit

warnings.filterwarnings("ignore")

RANDOM_STATE = 42
MICRO_AREA_SUPPORT_THRESHOLD = 30

PROJECT_ROOT = Path("/content/drive/MyDrive/DhakaNest_ML")

RAW_DIR = PROJECT_ROOT / "data" / "raw"
INTERIM_DIR = PROJECT_ROOT / "data" / "interim"
PROCESSED_DIR = PROJECT_ROOT / "data" / "processed"
SPLITS_DIR = PROJECT_ROOT / "data" / "splits"

REPORTS_DIR = PROJECT_ROOT / "reports"
AUDIT_DIR = REPORTS_DIR / "dataset_audit"
FIGURES_DIR = REPORTS_DIR / "figures"
METRICS_DIR = REPORTS_DIR / "metrics"
SEGMENT_DIR = REPORTS_DIR / "segment_evaluation"

MANIFESTS_DIR = PROJECT_ROOT / "manifests"
LOGS_DIR = PROJECT_ROOT / "logs"

RAW_DATA_PATH = RAW_DIR / "houserentdhaka.csv"

for directory in [
    INTERIM_DIR,
    PROCESSED_DIR,
    SPLITS_DIR,
    AUDIT_DIR,
    FIGURES_DIR,
    METRICS_DIR,
    SEGMENT_DIR,
    MANIFESTS_DIR,
    LOGS_DIR,
]:
    directory.mkdir(parents=True, exist_ok=True)

print("Project root:", PROJECT_ROOT)
print("Raw dataset:", RAW_DATA_PATH)
print("Random state:", RANDOM_STATE)
print("Micro-area threshold:", MICRO_AREA_SUPPORT_THRESHOLD)

Project root: /content/drive/MyDrive/DhakaNest_ML
Raw dataset: /content/drive/MyDrive/DhakaNest_ML/data/raw/houserentdhaka.csv
Random state: 42
Micro-area threshold: 30


## Dataset Audit

The original CSV is loaded without modifying the file stored in the raw-data
directory.

In [15]:
EXPECTED_COLUMNS = [
    "Unnamed: 0",
    "Location",
    "Area",
    "Bed",
    "Bath",
    "Price",
]

EXPECTED_SHAPE = (28800, 6)

if not RAW_DATA_PATH.exists():
    raise FileNotFoundError(
        f"Dataset was not found at: {RAW_DATA_PATH}"
    )

raw_df = pd.read_csv(RAW_DATA_PATH)

assert raw_df.shape == EXPECTED_SHAPE, (
    f"Expected shape {EXPECTED_SHAPE}, but found {raw_df.shape}"
)

assert raw_df.columns.tolist() == EXPECTED_COLUMNS, (
    f"Unexpected columns: {raw_df.columns.tolist()}"
)

print("Dataset shape:", raw_df.shape)
print("Columns:", raw_df.columns.tolist())

display(raw_df.head())

Dataset shape: (28800, 6)
Columns: ['Unnamed: 0', 'Location', 'Area', 'Bed', 'Bath', 'Price']


,Unnamed: 0,Location,Area,Bed,Bath,Price
0,0,"Block H, Bashundhara R-A, Dhaka","1,600 sqft",3,3,20 Thousand
1,1,"Farmgate, Tejgaon, Dhaka",900 sqft,2,2,20 Thousand
2,2,"Block B, Nobodoy Housing Society, Mohammadpur,...","1,250 sqft",3,3,18 Thousand
3,3,"Gulshan 1, Gulshan, Dhaka","2,200 sqft",3,4,75 Thousand
4,4,"Baridhara, Dhaka","2,200 sqft",3,3,75 Thousand


In [16]:
print("DATA TYPES")
display(raw_df.dtypes.to_frame("dtype"))

print("\nMISSING VALUES")
missing_summary = pd.DataFrame({
    "missing_count": raw_df.isna().sum(),
    "missing_percentage": (
        raw_df.isna().mean() * 100
    ).round(4),
})

display(missing_summary)

print("\nUNIQUE VALUES")
unique_summary = pd.DataFrame({
    "unique_count": raw_df.nunique(dropna=True)
})

display(unique_summary)

print("\nRAW LOCATION COUNT:", raw_df["Location"].nunique(dropna=True))
print("RAW AREA COUNT:", raw_df["Area"].nunique(dropna=True))
print("RAW PRICE COUNT:", raw_df["Price"].nunique(dropna=True))

DATA TYPES


,dtype
Unnamed: 0,int64
Location,object
Area,object
Bed,int64
Bath,int64
Price,object



MISSING VALUES


,missing_count,missing_percentage
Unnamed: 0,0,0.0
Location,0,0.0
Area,0,0.0
Bed,0,0.0
Bath,0,0.0
Price,0,0.0



UNIQUE VALUES


,unique_count
Unnamed: 0,4800
Location,730
Area,596
Bed,6
Bath,7
Price,223



RAW LOCATION COUNT: 730
RAW AREA COUNT: 596
RAW PRICE COUNT: 223


In [17]:
profile_rows = []

for column in raw_df.columns:
    non_null_values = raw_df[column].dropna()

    sample_values = (
        non_null_values.astype(str)
        .drop_duplicates()
        .head(8)
        .tolist()
    )

    profile_rows.append({
        "column": column,
        "dtype": str(raw_df[column].dtype),
        "row_count": len(raw_df),
        "missing_count": int(raw_df[column].isna().sum()),
        "missing_percentage": round(
            raw_df[column].isna().mean() * 100, 4
        ),
        "unique_count": int(raw_df[column].nunique(dropna=True)),
        "sample_values": " | ".join(sample_values),
    })

raw_profile_df = pd.DataFrame(profile_rows)

RAW_PROFILE_PATH = AUDIT_DIR / "raw_data_profile.csv"
raw_profile_df.to_csv(RAW_PROFILE_PATH, index=False)

display(raw_profile_df)
print("Saved:", RAW_PROFILE_PATH)

,column,dtype,row_count,missing_count,missing_percentage,unique_count,sample_values
0,Unnamed: 0,int64,28800,0,0.0,4800,0 | 1 | 2 | 3 | 4 | 5 | 6 | 7
1,Location,object,28800,0,0.0,730,"Block H, Bashundhara R-A, Dhaka | Farmgate, Te..."
2,Area,object,28800,0,0.0,596,"1,600 sqft | 900 sqft | 1,250 sqft | 2,200 sqf..."
3,Bed,int64,28800,0,0.0,6,3 | 2 | 4 | 1 | 5 | 6
4,Bath,int64,28800,0,0.0,7,3 | 2 | 4 | 5 | 1 | 6 | 8
5,Price,object,28800,0,0.0,223,20 Thousand | 18 Thousand | 75 Thousand | 50 T...


Saved: /content/drive/MyDrive/DhakaNest_ML/reports/dataset_audit/raw_data_profile.csv


In [18]:
location_frequency = (
    raw_df["Location"]
    .fillna("<MISSING>")
    .astype(str)
    .value_counts(dropna=False)
    .rename_axis("location_raw")
    .reset_index(name="record_count")
)

area_frequency = (
    raw_df["Area"]
    .fillna("<MISSING>")
    .astype(str)
    .value_counts(dropna=False)
    .rename_axis("area_raw")
    .reset_index(name="record_count")
)

price_frequency = (
    raw_df["Price"]
    .fillna("<MISSING>")
    .astype(str)
    .value_counts(dropna=False)
    .rename_axis("price_raw")
    .reset_index(name="record_count")
)

location_frequency.to_csv(
    AUDIT_DIR / "raw_location_frequency.csv",
    index=False,
)

area_frequency.to_csv(
    AUDIT_DIR / "raw_area_frequency.csv",
    index=False,
)

price_frequency.to_csv(
    AUDIT_DIR / "raw_price_frequency.csv",
    index=False,
)

print("Most frequent locations:")
display(location_frequency.head(20))

print("Area examples:")
display(area_frequency.head(20))

print("Price examples:")
display(price_frequency.head(20))

Most frequent locations:


,location_raw,record_count
0,"Mohammadpur, Dhaka",757
1,"Mirpur, Dhaka",556
2,"Block D, Section 12, Mirpur, Dhaka",417
3,"Dhanmondi, Dhaka",414
4,"Block E, Section 12, Mirpur, Dhaka",411
5,"Sector 10, Uttara, Dhaka",357
6,"Paikpara, Ahmed Nagar, Mirpur, Dhaka",352
7,"Kallyanpur, Mirpur, Dhaka",337
8,"Section 12, Mirpur, Dhaka",311
9,"Block B, Section 12, Mirpur, Dhaka",307


Area examples:


,area_raw,record_count
0,650 sqft,3042
1,700 sqft,2810
2,800 sqft,2396
3,"1,200 sqft",1591
4,750 sqft,1394
5,"1,100 sqft",1376
6,"1,000 sqft",1155
7,600 sqft,1092
8,900 sqft,1086
9,850 sqft,799


Price examples:


,price_raw,record_count
0,15 Thousand,2679
1,12 Thousand,2056
2,20 Thousand,2019
3,16 Thousand,1947
4,14 Thousand,1875
5,18 Thousand,1763
6,13 Thousand,1671
7,10 Thousand,1663
8,11 Thousand,1370
9,25 Thousand,1234


In [19]:
MEANINGFUL_RAW_COLUMNS = [
    "Location",
    "Area",
    "Bed",
    "Bath",
    "Price",
]

exact_duplicate_mask = raw_df.duplicated(
    subset=MEANINGFUL_RAW_COLUMNS,
    keep=False,
)

exact_duplicate_count = int(exact_duplicate_mask.sum())

duplicate_groups_count = int(
    raw_df.loc[exact_duplicate_mask, MEANINGFUL_RAW_COLUMNS]
    .drop_duplicates()
    .shape[0]
)

print("Rows belonging to exact duplicate groups:", exact_duplicate_count)
print("Number of exact duplicate groups:", duplicate_groups_count)

display(
    raw_df.loc[exact_duplicate_mask, MEANINGFUL_RAW_COLUMNS]
    .sort_values(MEANINGFUL_RAW_COLUMNS)
    .head(30)
)

Rows belonging to exact duplicate groups: 17878
Number of exact duplicate groups: 4337


,Location,Area,Bed,Bath,Price
11005,"1st Colony, Mirpur, Dhaka","1,100 sqft",3,3,15 Thousand
21793,"1st Colony, Mirpur, Dhaka","1,100 sqft",3,3,15 Thousand
2705,"1st Colony, Mirpur, Dhaka","1,100 sqft",3,3,20 Thousand
2706,"1st Colony, Mirpur, Dhaka","1,100 sqft",3,3,20 Thousand
215,"1st Colony, Mirpur, Dhaka","1,200 sqft",3,3,16 Thousand
16090,"1st Colony, Mirpur, Dhaka","1,200 sqft",3,3,16 Thousand
14975,"1st Colony, Mirpur, Dhaka","1,200 sqft",3,3,17 Thousand
22359,"1st Colony, Mirpur, Dhaka","1,200 sqft",3,3,17 Thousand
25972,"1st Colony, Mirpur, Dhaka","1,200 sqft",3,3,17 Thousand
22358,"1st Colony, Mirpur, Dhaka","1,300 sqft",3,3,18 Thousand


In [22]:
# Capture the output of raw_df.info()
buffer = StringIO()
raw_df.info(buf=buffer)
dataframe_info = buffer.getvalue()

# Convert summary tables into plain text
missing_summary_text = missing_summary.to_string()
unique_summary_text = unique_summary.to_string()

# Build the Markdown report line by line.
# This avoids problems with nested triple quotes and Markdown code fences.
report_lines = [
    "# DhakaNest Raw Dataset Audit",
    "",
    "## Audit Information",
    "",
    f"- Audit time (UTC): {datetime.now(timezone.utc).isoformat()}",
    f"- Source file: `{RAW_DATA_PATH.name}`",
    f"- Number of rows: {raw_df.shape[0]:,}",
    f"- Number of columns: {raw_df.shape[1]}",
    (
        "- Exact duplicated rows within meaningful columns: "
        f"{exact_duplicate_count:,}"
    ),
    f"- Exact duplicate groups: {duplicate_groups_count:,}",
    (
        "- Unique raw locations: "
        f"{raw_df['Location'].nunique(dropna=True):,}"
    ),
    "",
    "## Original Columns",
    "",
    ", ".join(raw_df.columns.tolist()),
    "",
    "## Missing Values",
    "",
    "```text",
    missing_summary_text,
    "```",
    "",
    "## Unique Values",
    "",
    "```text",
    unique_summary_text,
    "```",
    "",
    "## DataFrame Information",
    "",
    "```text",
    dataframe_info,
    "```",
    "",
    "## Initial Decisions",
    "",
    "- `Unnamed: 0` will not be used as a model feature.",
    "- The original CSV will remain unchanged.",
    "- `Area` will be parsed into numeric square feet.",
    "- `Price` will be converted into numeric BDT.",
    "- `Bed` and `Bath` will be converted into whole-number values.",
    "- Exact source duplicates will be investigated and removed.",
    (
        "- Repeated normalized property combinations will be "
        "retained but grouped."
    ),
    "- Malformed and invalid records will be quarantined.",
    (
        "- Valid statistical outliers will be flagged rather "
        "than automatically deleted."
    ),
    (
        "- Rent per square foot may be used for analysis, "
        "but not as a model input."
    ),
    "",
]

dataset_audit_text = "\n".join(report_lines)

DATASET_AUDIT_PATH = AUDIT_DIR / "dataset_audit.md"

DATASET_AUDIT_PATH.write_text(
    dataset_audit_text,
    encoding="utf-8",
)

print("Initial dataset audit report created successfully.")
print("Saved to:", DATASET_AUDIT_PATH)

Initial dataset audit report created successfully.
Saved to: /content/drive/MyDrive/DhakaNest_ML/reports/dataset_audit/dataset_audit.md


In [23]:
assert DATASET_AUDIT_PATH.exists(), (
    f"Audit report was not created: {DATASET_AUDIT_PATH}"
)

saved_audit_text = DATASET_AUDIT_PATH.read_text(
    encoding="utf-8"
)

required_sections = [
    "# DhakaNest Raw Dataset Audit",
    "## Audit Information",
    "## Missing Values",
    "## Unique Values",
    "## DataFrame Information",
    "## Initial Decisions",
]

for section in required_sections:
    assert section in saved_audit_text, (
        f"Missing report section: {section}"
    )

print("Audit report verification passed.")
print("\nReport preview:\n")
print(saved_audit_text[:2000])

Audit report verification passed.

Report preview:

# DhakaNest Raw Dataset Audit

## Audit Information

- Audit time (UTC): 2026-07-30T18:31:19.626054+00:00
- Source file: `houserentdhaka.csv`
- Number of rows: 28,800
- Number of columns: 6
- Exact duplicated rows within meaningful columns: 17,878
- Exact duplicate groups: 4,337
- Unique raw locations: 730

## Original Columns

Unnamed: 0, Location, Area, Bed, Bath, Price

## Missing Values

```text
            missing_count  missing_percentage
Unnamed: 0              0                 0.0
Location                0                 0.0
Area                    0                 0.0
Bed                     0                 0.0
Bath                    0                 0.0
Price                   0                 0.0
```

## Unique Values

```text
            unique_count
Unnamed: 0          4800
Location             730
Area                 596
Bed                    6
Bath                   7
Price                223
```

## DataFrame

# Part 3 — Parsing and Standardization

The raw text values are transformed into numeric, model-ready values.

The parsing process follows these rules:

- Malformed values are returned as missing values.
- Invalid values are never silently converted to zero.
- Unsupported land-area units are not converted automatically.
- Rent values expressed in Thousand, Lakh, Lac, or Crore are converted to BDT.
- Bedroom and bathroom values must represent whole numbers.
- Positivity and validity are checked later during record validation.

In [24]:
def extract_first_number(value):
    """
    Extract the first signed integer or decimal number from a value.

    Examples:
        "1,200 sqft"     -> 1200.0
        "18.63 Thousand" -> 18.63
        3                -> 3.0

    Returns:
        float or np.nan
    """
    if pd.isna(value):
        return np.nan

    if isinstance(
        value,
        (int, float, np.integer, np.floating),
    ):
        return float(value)

    text = str(value).strip().lower()

    # Remove commas used as thousands separators.
    text = text.replace(",", "")

    match = re.search(
        r"[-+]?\d+(?:\.\d+)?",
        text,
    )

    if match is None:
        return np.nan

    try:
        return float(match.group())
    except (TypeError, ValueError):
        return np.nan


def parse_area_sqft(value):
    """
    Parse a property-area value expressed in square feet.

    Examples:
        "1,200 sqft" -> 1200.0
        "950 Sq. Ft." -> 950.0
        1500 -> 1500.0

    Unsupported land units are returned as np.nan because an explicit,
    academically justified conversion rule would be required.
    """
    if pd.isna(value):
        return np.nan

    text = str(value).strip().lower()

    unsupported_units = {
        "katha",
        "kattha",
        "decimal",
        "acre",
        "bigha",
    }

    if any(
        unit in text
        for unit in unsupported_units
    ):
        return np.nan

    number = extract_first_number(value)

    if pd.isna(number):
        return np.nan

    return float(number)


def parse_price_bdt(value):
    """
    Convert a raw rent value into Bangladeshi Taka.

    Supported examples:
        "20 Thousand"    -> 20000.0
        "18.63 Thousand" -> 18630.0
        "1.6 Lakh"       -> 160000.0
        "1.6 Lac"        -> 160000.0
        "25k"            -> 25000.0
        "25,000"         -> 25000.0
        "1 Crore"        -> 10000000.0

    Returns:
        float or np.nan
    """
    if pd.isna(value):
        return np.nan

    if isinstance(
        value,
        (int, float, np.integer, np.floating),
    ):
        return float(value)

    text = str(value).strip().lower()

    # Remove common currency markers.
    text = text.replace("৳", "")
    text = re.sub(r"\bbdt\b", "", text)
    text = re.sub(r"\btaka\b", "", text)

    number = extract_first_number(text)

    if pd.isna(number):
        return np.nan

    if re.search(r"\b(?:crore|cr)\b", text):
        multiplier = 10_000_000

    elif re.search(r"\b(?:lakh|lac)\b", text):
        multiplier = 100_000

    elif (
        re.search(r"\bthousand\b", text)
        or re.search(r"\d+(?:\.\d+)?\s*k\b", text)
    ):
        multiplier = 1_000

    else:
        multiplier = 1

    return float(number * multiplier)


def parse_positive_integer(value):
    """
    Parse a value only when it represents a whole number.

    Positivity is checked later during invalid-record validation so that
    zero and negative values can receive a clear invalidity reason.

    Examples:
        "3 Beds" -> 3.0
        2 -> 2.0
        "2.5" -> np.nan
    """
    number = extract_first_number(value)

    if pd.isna(number):
        return np.nan

    rounded_number = round(number)

    if not np.isclose(
        number,
        rounded_number,
    ):
        return np.nan

    return float(rounded_number)


print("Parsing functions defined successfully.")

Parsing functions defined successfully.


In [25]:
area_test_cases = {
    "1,200 sqft": 1200.0,
    "950 Sq. Ft.": 950.0,
    "1500": 1500.0,
    800: 800.0,
    "5 Katha": np.nan,
    None: np.nan,
}

price_test_cases = {
    "20 Thousand": 20000.0,
    "18.63 Thousand": 18630.0,
    "1.6 Lakh": 160000.0,
    "1.6 Lac": 160000.0,
    "25k": 25000.0,
    "BDT 25,000": 25000.0,
    "৳30,000": 30000.0,
    30000: 30000.0,
    None: np.nan,
}

integer_test_cases = {
    "3 Beds": 3.0,
    "2 Bathrooms": 2.0,
    4: 4.0,
    "2.5": np.nan,
    None: np.nan,
}


def values_match(actual, expected):
    if pd.isna(expected):
        return pd.isna(actual)

    return np.isclose(actual, expected)


print("AREA PARSING TESTS")
for raw_value, expected_value in area_test_cases.items():
    actual_value = parse_area_sqft(raw_value)
    passed = values_match(actual_value, expected_value)

    print(
        f"{raw_value!r:20} -> "
        f"{actual_value!r:12} "
        f"{'PASS' if passed else 'FAIL'}"
    )

    assert passed, (
        f"Area parser failed for {raw_value!r}: "
        f"expected {expected_value}, found {actual_value}"
    )


print("\nPRICE PARSING TESTS")
for raw_value, expected_value in price_test_cases.items():
    actual_value = parse_price_bdt(raw_value)
    passed = values_match(actual_value, expected_value)

    print(
        f"{raw_value!r:20} -> "
        f"{actual_value!r:12} "
        f"{'PASS' if passed else 'FAIL'}"
    )

    assert passed, (
        f"Price parser failed for {raw_value!r}: "
        f"expected {expected_value}, found {actual_value}"
    )


print("\nINTEGER PARSING TESTS")
for raw_value, expected_value in integer_test_cases.items():
    actual_value = parse_positive_integer(raw_value)
    passed = values_match(actual_value, expected_value)

    print(
        f"{raw_value!r:20} -> "
        f"{actual_value!r:12} "
        f"{'PASS' if passed else 'FAIL'}"
    )

    assert passed, (
        f"Integer parser failed for {raw_value!r}: "
        f"expected {expected_value}, found {actual_value}"
    )


print("\nAll parsing tests passed successfully.")

AREA PARSING TESTS
'1,200 sqft'         -> 1200.0       PASS
'950 Sq. Ft.'        -> 950.0        PASS
'1500'               -> 1500.0       PASS
800                  -> 800.0        PASS
'5 Katha'            -> nan          PASS
None                 -> nan          PASS

PRICE PARSING TESTS
'20 Thousand'        -> 20000.0      PASS
'18.63 Thousand'     -> 18630.0      PASS
'1.6 Lakh'           -> 160000.0     PASS
'1.6 Lac'            -> 160000.0     PASS
'25k'                -> 25000.0      PASS
'BDT 25,000'         -> 25000.0      PASS
'৳30,000'            -> 30000.0      PASS
30000                -> 30000.0      PASS
None                 -> nan          PASS

INTEGER PARSING TESTS
'3 Beds'             -> 3.0          PASS
'2 Bathrooms'        -> 2.0          PASS
4                    -> 4.0          PASS
'2.5'                -> nan          PASS
None                 -> nan          PASS

All parsing tests passed successfully.


In [26]:
working_df = raw_df.copy()

working_df["record_id"] = [
    f"DN-{index + 1:06d}"
    for index in range(len(working_df))
]

working_df["location_raw"] = (
    working_df["Location"]
    .astype("string")
    .str.strip()
)

working_df["area_sqft"] = (
    working_df["Area"]
    .apply(parse_area_sqft)
)

working_df["bedrooms"] = (
    working_df["Bed"]
    .apply(parse_positive_integer)
)

working_df["bathrooms"] = (
    working_df["Bath"]
    .apply(parse_positive_integer)
)

working_df["base_rent_bdt"] = (
    working_df["Price"]
    .apply(parse_price_bdt)
)

working_df["area_parse_ok"] = working_df["area_sqft"].notna()
working_df["bedrooms_parse_ok"] = working_df["bedrooms"].notna()
working_df["bathrooms_parse_ok"] = working_df["bathrooms"].notna()
working_df["rent_parse_ok"] = working_df["base_rent_bdt"].notna()

parsed_preview_columns = [
    "record_id",
    "Location",
    "Area",
    "area_sqft",
    "Bed",
    "bedrooms",
    "Bath",
    "bathrooms",
    "Price",
    "base_rent_bdt",
]

display(working_df[parsed_preview_columns].head(20))

,record_id,Location,Area,area_sqft,Bed,bedrooms,Bath,bathrooms,Price,base_rent_bdt
0,DN-000001,"Block H, Bashundhara R-A, Dhaka","1,600 sqft",1600.0,3,3.0,3,3.0,20 Thousand,20000.0
1,DN-000002,"Farmgate, Tejgaon, Dhaka",900 sqft,900.0,2,2.0,2,2.0,20 Thousand,20000.0
2,DN-000003,"Block B, Nobodoy Housing Society, Mohammadpur,...","1,250 sqft",1250.0,3,3.0,3,3.0,18 Thousand,18000.0
3,DN-000004,"Gulshan 1, Gulshan, Dhaka","2,200 sqft",2200.0,3,3.0,4,4.0,75 Thousand,75000.0
4,DN-000005,"Baridhara, Dhaka","2,200 sqft",2200.0,3,3.0,3,3.0,75 Thousand,75000.0
5,DN-000006,"Bashundhara R-A, Dhaka","3,000 sqft",3000.0,4,4.0,5,5.0,50 Thousand,50000.0
6,DN-000007,"Baridhara, Dhaka","2,300 sqft",2300.0,3,3.0,3,3.0,75 Thousand,75000.0
7,DN-000008,"PC Culture Housing, Mohammadpur, Dhaka",950 sqft,950.0,2,2.0,2,2.0,14 Thousand,14000.0
8,DN-000009,"Jigatola, Hazaribag, Dhaka","1,600 sqft",1600.0,3,3.0,3,3.0,28 Thousand,28000.0
9,DN-000010,"West Kazipara, Mirpur, Dhaka","1,150 sqft",1150.0,3,3.0,3,3.0,19 Thousand,19000.0


In [27]:
parsing_failure_mask = ~(
    working_df["area_parse_ok"]
    & working_df["bedrooms_parse_ok"]
    & working_df["bathrooms_parse_ok"]
    & working_df["rent_parse_ok"]
)

parsing_failures_df = working_df.loc[
    parsing_failure_mask,
    [
        "record_id",
        "Location",
        "Area",
        "Bed",
        "Bath",
        "Price",
        "area_sqft",
        "bedrooms",
        "bathrooms",
        "base_rent_bdt",
        "area_parse_ok",
        "bedrooms_parse_ok",
        "bathrooms_parse_ok",
        "rent_parse_ok",
    ],
].copy()

PARSING_FAILURE_PATH = INTERIM_DIR / "parsing_failures.csv"
parsing_failures_df.to_csv(
    PARSING_FAILURE_PATH,
    index=False,
)

print("Parsing-failure rows:", len(parsing_failures_df))
print("Saved:", PARSING_FAILURE_PATH)

display(parsing_failures_df.head(30))

Parsing-failure rows: 0
Saved: /content/drive/MyDrive/DhakaNest_ML/data/interim/parsing_failures.csv


,record_id,Location,Area,Bed,Bath,Price,area_sqft,bedrooms,bathrooms,base_rent_bdt,area_parse_ok,bedrooms_parse_ok,bathrooms_parse_ok,rent_parse_ok


## Location Normalization

Location strings are separated into broad area, micro-area, and optional
sub-area detail. Automatic normalization is followed by an optional manual
override mechanism.

In [28]:
LOCATION_SUFFIXES = {
    "dhaka",
    "dhaka city",
    "bangladesh",
    "dhaka bangladesh",
}


def normalize_unicode_text(value):
    if pd.isna(value):
        return ""

    text = unicodedata.normalize("NFKC", str(value))
    text = re.sub(r"\s+", " ", text).strip()

    return text


def location_key(value):
    text = normalize_unicode_text(value).lower()
    text = text.replace("&", " and ")
    text = re.sub(r"[^a-z0-9]+", " ", text)
    text = re.sub(r"\s+", " ", text).strip()

    return text


def format_location_component(value):
    text = normalize_unicode_text(value)

    text = re.sub(r"\s*-\s*", " ", text)
    text = re.sub(r"\s+", " ", text).strip(" ,;-")

    formatted = text.title()

    replacements = {
        "Dohs": "DOHS",
        "R/A": "R/A",
        "R A": "R/A",
        "Rd": "Road",
    }

    for old, new in replacements.items():
        formatted = formatted.replace(old, new)

    return formatted


def is_micro_area_like(value):
    key = location_key(value)

    patterns = [
        r"^section\s*\d+",
        r"^sector\s*\d+",
        r"^block\s*[a-z0-9]+",
        r"^road\s*\d+",
        r"^lane\s*\d+",
        r"^avenue\s*\d+",
        r"^phase\s*\d+",
    ]

    return any(re.search(pattern, key) for pattern in patterns)


def split_embedded_broad_micro(component):
    """
    Handle forms such as:
    Mirpur 10
    Mirpur Section 10
    Uttara Sector 10
    """
    component = format_location_component(component)

    mirpur_match = re.match(
        r"^Mirpur\s*(?:Section\s*)?(\d+[A-Za-z]?)$",
        component,
        flags=re.IGNORECASE,
    )

    if mirpur_match:
        section = mirpur_match.group(1).upper()
        return "Mirpur", f"Section {section}"

    generic_match = re.match(
        r"^(.+?)\s+(Sector|Section)\s*(\d+[A-Za-z]?)$",
        component,
        flags=re.IGNORECASE,
    )

    if generic_match:
        broad = format_location_component(generic_match.group(1))
        label = generic_match.group(2).title()
        number = generic_match.group(3).upper()

        return broad, f"{label} {number}"

    return None


def normalize_location(value):
    text = normalize_unicode_text(value)

    if not text:
        return {
            "broad_area": "Unknown",
            "micro_area": "Unknown",
            "sub_area_detail": "",
            "normalization_confidence": "low",
        }

    raw_parts = re.split(r"[,;|]+", text)

    parts = [
        format_location_component(part)
        for part in raw_parts
        if format_location_component(part)
    ]

    while parts and location_key(parts[-1]) in LOCATION_SUFFIXES:
        parts.pop()

    if not parts:
        return {
            "broad_area": "Unknown",
            "micro_area": "Unknown",
            "sub_area_detail": "",
            "normalization_confidence": "low",
        }

    if len(parts) == 1:
        embedded = split_embedded_broad_micro(parts[0])

        if embedded:
            broad_area, micro_area = embedded
        else:
            broad_area = parts[0]
            micro_area = parts[0]

        return {
            "broad_area": broad_area,
            "micro_area": micro_area,
            "sub_area_detail": "",
            "normalization_confidence": "medium",
        }

    broad_candidate = parts[-1]
    micro_candidate = parts[-2]
    sub_parts = parts[:-2]

    # Correct reversed forms such as:
    # Uttara, Sector 10, Dhaka
    if (
        is_micro_area_like(broad_candidate)
        and not is_micro_area_like(micro_candidate)
    ):
        broad_area = micro_candidate
        micro_area = broad_candidate
    else:
        broad_area = broad_candidate
        micro_area = micro_candidate

    return {
        "broad_area": format_location_component(broad_area),
        "micro_area": format_location_component(micro_area),
        "sub_area_detail": ", ".join(sub_parts),
        "normalization_confidence": "high",
    }

In [29]:
location_results = (
    working_df["location_raw"]
    .apply(normalize_location)
    .apply(pd.Series)
)

working_df = pd.concat(
    [
        working_df,
        location_results,
    ],
    axis=1,
)

working_df["location_alias_key"] = (
    working_df["location_raw"]
    .apply(location_key)
)

location_reference_df = (
    working_df.groupby(
        [
            "location_raw",
            "location_alias_key",
            "broad_area",
            "micro_area",
            "sub_area_detail",
            "normalization_confidence",
        ],
        dropna=False,
    )
    .size()
    .reset_index(name="record_count")
    .sort_values(
        ["record_count", "location_raw"],
        ascending=[False, True],
    )
)

LOCATION_REFERENCE_PATH = PROCESSED_DIR / "location_reference.csv"
location_reference_df.to_csv(
    LOCATION_REFERENCE_PATH,
    index=False,
)

print("Unique normalized broad areas:", working_df["broad_area"].nunique())
print("Unique normalized micro-areas:", working_df["micro_area"].nunique())

display(location_reference_df.head(50))

Unique normalized broad areas: 69
Unique normalized micro-areas: 354


,location_raw,location_alias_key,broad_area,micro_area,sub_area_detail,normalization_confidence,record_count
462,"Mohammadpur, Dhaka",mohammadpur dhaka,Mohammadpur,Mohammadpur,,medium,757
452,"Mirpur, Dhaka",mirpur dhaka,Mirpur,Mirpur,,medium,556
194,"Block D, Section 12, Mirpur, Dhaka",block d section 12 mirpur dhaka,Mirpur,Section 12,Block D,high,417
297,"Dhanmondi, Dhaka",dhanmondi dhaka,Dhanmondi,Dhanmondi,,medium,414
212,"Block E, Section 12, Mirpur, Dhaka",block e section 12 mirpur dhaka,Mirpur,Section 12,Block E,high,411
591,"Sector 10, Uttara, Dhaka",sector 10 uttara dhaka,Uttara,Sector 10,,high,357
521,"Paikpara, Ahmed Nagar, Mirpur, Dhaka",paikpara ahmed nagar mirpur dhaka,Mirpur,Ahmed Nagar,Paikpara,high,352
386,"Kallyanpur, Mirpur, Dhaka",kallyanpur mirpur dhaka,Mirpur,Kallyanpur,,high,337
579,"Section 12, Mirpur, Dhaka",section 12 mirpur dhaka,Mirpur,Section 12,,high,311
132,"Block B, Section 12, Mirpur, Dhaka",block b section 12 mirpur dhaka,Mirpur,Section 12,Block B,high,307


In [30]:
unknown_location_mask = (
    working_df["broad_area"].eq("Unknown")
    | working_df["micro_area"].eq("Unknown")
)

alias_conflict_summary = (
    location_reference_df.groupby("location_alias_key")
    .agg(
        broad_area_versions=("broad_area", "nunique"),
        micro_area_versions=("micro_area", "nunique"),
        raw_versions=("location_raw", "nunique"),
        record_count=("record_count", "sum"),
    )
    .reset_index()
)

conflicting_alias_keys = set(
    alias_conflict_summary.loc[
        (alias_conflict_summary["broad_area_versions"] > 1)
        | (alias_conflict_summary["micro_area_versions"] > 1),
        "location_alias_key",
    ]
)

unresolved_locations_df = location_reference_df.loc[
    location_reference_df["location_alias_key"].isin(conflicting_alias_keys)
    | location_reference_df["broad_area"].eq("Unknown")
    | location_reference_df["micro_area"].eq("Unknown")
].copy()

UNRESOLVED_LOCATIONS_PATH = INTERIM_DIR / "unresolved_locations.csv"
unresolved_locations_df.to_csv(
    UNRESOLVED_LOCATIONS_PATH,
    index=False,
)

print("Unresolved or conflicting location rows:", len(unresolved_locations_df))
print("Saved:", UNRESOLVED_LOCATIONS_PATH)

display(unresolved_locations_df.head(50))

Unresolved or conflicting location rows: 1
Saved: /content/drive/MyDrive/DhakaNest_ML/data/interim/unresolved_locations.csv


,location_raw,location_alias_key,broad_area,micro_area,sub_area_detail,normalization_confidence,record_count
294,Dhaka,dhaka,Unknown,Unknown,,low,13


In [31]:
MANUAL_OVERRIDE_PATH = INTERIM_DIR / "manual_location_overrides.csv"

if not MANUAL_OVERRIDE_PATH.exists():
    manual_override_template = location_reference_df[
        [
            "location_raw",
            "record_count",
            "broad_area",
            "micro_area",
            "sub_area_detail",
            "normalization_confidence",
        ]
    ].copy()

    manual_override_template = manual_override_template.rename(
        columns={
            "broad_area": "automatic_broad_area",
            "micro_area": "automatic_micro_area",
            "sub_area_detail": "automatic_sub_area_detail",
        }
    )

    manual_override_template["broad_area_override"] = ""
    manual_override_template["micro_area_override"] = ""
    manual_override_template["sub_area_detail_override"] = ""
    manual_override_template["review_note"] = ""

    manual_override_template.to_csv(
        MANUAL_OVERRIDE_PATH,
        index=False,
    )

    print("Created manual override template.")
else:
    print("Existing manual override file preserved.")

print("File:", MANUAL_OVERRIDE_PATH)

Created manual override template.
File: /content/drive/MyDrive/DhakaNest_ML/data/interim/manual_location_overrides.csv


In [32]:
override_df = pd.read_csv(
    MANUAL_OVERRIDE_PATH,
    dtype=str,
).fillna("")

override_df["location_raw"] = (
    override_df["location_raw"]
    .astype(str)
    .str.strip()
)

override_lookup = override_df.set_index("location_raw").to_dict("index")


def apply_location_override(row):
    raw_location = str(row["location_raw"]).strip()

    override = override_lookup.get(raw_location)

    if not override:
        return row

    broad_override = override.get(
        "broad_area_override", ""
    ).strip()

    micro_override = override.get(
        "micro_area_override", ""
    ).strip()

    detail_override = override.get(
        "sub_area_detail_override", ""
    ).strip()

    changed = False

    if broad_override:
        row["broad_area"] = format_location_component(broad_override)
        changed = True

    if micro_override:
        row["micro_area"] = format_location_component(micro_override)
        changed = True

    if detail_override:
        row["sub_area_detail"] = format_location_component(detail_override)
        changed = True

    if changed:
        row["normalization_confidence"] = "manual"

    return row


working_df = working_df.apply(
    apply_location_override,
    axis=1,
)

print(
    working_df["normalization_confidence"]
    .value_counts(dropna=False)
)

normalization_confidence
high      22554
medium     6233
low          13
Name: count, dtype: int64


In [33]:
final_location_reference_df = (
    working_df.groupby(
        [
            "location_raw",
            "location_alias_key",
            "broad_area",
            "micro_area",
            "sub_area_detail",
            "normalization_confidence",
        ],
        dropna=False,
    )
    .size()
    .reset_index(name="record_count")
    .sort_values(
        ["record_count", "location_raw"],
        ascending=[False, True],
    )
)

location_aliases_df = final_location_reference_df[
    [
        "location_alias_key",
        "location_raw",
        "broad_area",
        "micro_area",
        "normalization_confidence",
        "record_count",
    ]
].copy()

location_aliases_df.to_csv(
    PROCESSED_DIR / "location_aliases.csv",
    index=False,
)

final_location_reference_df.to_csv(
    PROCESSED_DIR / "location_reference.csv",
    index=False,
)

print("Final location files saved.")
display(final_location_reference_df.head(30))

Final location files saved.


,location_raw,location_alias_key,broad_area,micro_area,sub_area_detail,normalization_confidence,record_count
462,"Mohammadpur, Dhaka",mohammadpur dhaka,Mohammadpur,Mohammadpur,,medium,757
452,"Mirpur, Dhaka",mirpur dhaka,Mirpur,Mirpur,,medium,556
194,"Block D, Section 12, Mirpur, Dhaka",block d section 12 mirpur dhaka,Mirpur,Section 12,Block D,high,417
297,"Dhanmondi, Dhaka",dhanmondi dhaka,Dhanmondi,Dhanmondi,,medium,414
212,"Block E, Section 12, Mirpur, Dhaka",block e section 12 mirpur dhaka,Mirpur,Section 12,Block E,high,411
591,"Sector 10, Uttara, Dhaka",sector 10 uttara dhaka,Uttara,Sector 10,,high,357
521,"Paikpara, Ahmed Nagar, Mirpur, Dhaka",paikpara ahmed nagar mirpur dhaka,Mirpur,Ahmed Nagar,Paikpara,high,352
386,"Kallyanpur, Mirpur, Dhaka",kallyanpur mirpur dhaka,Mirpur,Kallyanpur,,high,337
579,"Section 12, Mirpur, Dhaka",section 12 mirpur dhaka,Mirpur,Section 12,,high,311
132,"Block B, Section 12, Mirpur, Dhaka",block b section 12 mirpur dhaka,Mirpur,Section 12,Block B,high,307


## Duplicate Investigation

Exact source duplicates are removed. Repeated normalized combinations are
retained but assigned the same duplicate-group identifier.

In [34]:
working_df["raw_duplicate_rank"] = (
    working_df.groupby(
        MEANINGFUL_RAW_COLUMNS,
        dropna=False,
    )
    .cumcount()
)

working_df["raw_duplicate_group_size"] = (
    working_df.groupby(
        MEANINGFUL_RAW_COLUMNS,
        dropna=False,
    )["record_id"]
    .transform("size")
)

working_df["is_exact_source_duplicate"] = (
    working_df["raw_duplicate_rank"] > 0
)

confirmed_duplicate_rows_df = working_df.loc[
    working_df["raw_duplicate_group_size"] > 1
].copy()

confirmed_duplicate_rows_df["duplicate_action"] = np.where(
    confirmed_duplicate_rows_df["is_exact_source_duplicate"],
    "removed_exact_duplicate",
    "retained_first_occurrence",
)

CONFIRMED_DUPLICATES_PATH = (
    INTERIM_DIR / "confirmed_duplicate_rows.csv"
)

confirmed_duplicate_rows_df.to_csv(
    CONFIRMED_DUPLICATES_PATH,
    index=False,
)

print(
    "Exact duplicate rows to remove:",
    int(working_df["is_exact_source_duplicate"].sum()),
)

print(
    "Rows belonging to repeated source groups:",
    len(confirmed_duplicate_rows_df),
)

display(
    confirmed_duplicate_rows_df[
        [
            "record_id",
            "Location",
            "Area",
            "Bed",
            "Bath",
            "Price",
            "raw_duplicate_group_size",
            "duplicate_action",
        ]
    ].head(30)
)

Exact duplicate rows to remove: 13541
Rows belonging to repeated source groups: 17878


,record_id,Location,Area,Bed,Bath,Price,raw_duplicate_group_size,duplicate_action
1,DN-000002,"Farmgate, Tejgaon, Dhaka",900 sqft,2,2,20 Thousand,3,retained_first_occurrence
2,DN-000003,"Block B, Nobodoy Housing Society, Mohammadpur,...","1,250 sqft",3,3,18 Thousand,3,retained_first_occurrence
6,DN-000007,"Baridhara, Dhaka","2,300 sqft",3,3,75 Thousand,6,retained_first_occurrence
10,DN-000011,"Block D, Bashundhara R-A, Dhaka","1,300 sqft",3,3,40 Thousand,5,retained_first_occurrence
11,DN-000012,"Block J, Bashundhara R-A, Dhaka","1,250 sqft",3,3,19 Thousand,8,retained_first_occurrence
12,DN-000013,"Block D, Bashundhara R-A, Dhaka","1,300 sqft",3,3,60 Thousand,6,retained_first_occurrence
15,DN-000016,"Block D, Bashundhara R-A, Dhaka","1,300 sqft",3,3,35 Thousand,3,retained_first_occurrence
18,DN-000019,"Ibrahimpur, Dhaka","1,050 sqft",3,3,25 Thousand,2,retained_first_occurrence
19,DN-000020,"Ibrahimpur, Dhaka","1,050 sqft",3,3,25 Thousand,2,removed_exact_duplicate
22,DN-000023,"Block D, Bashundhara R-A, Dhaka","2,100 sqft",3,4,40 Thousand,6,retained_first_occurrence


In [35]:
source_unique_df = working_df.loc[
    ~working_df["is_exact_source_duplicate"]
].copy()

CLEANED_UNIQUE_PATH = INTERIM_DIR / "cleaned_unique_records.csv"

source_unique_df.to_csv(
    CLEANED_UNIQUE_PATH,
    index=False,
)

print("Rows before exact deduplication:", len(working_df))
print("Rows after exact deduplication:", len(source_unique_df))
print("Removed:", len(working_df) - len(source_unique_df))
print("Saved:", CLEANED_UNIQUE_PATH)

Rows before exact deduplication: 28800
Rows after exact deduplication: 15259
Removed: 13541
Saved: /content/drive/MyDrive/DhakaNest_ML/data/interim/cleaned_unique_records.csv


In [36]:
NORMALIZED_FINGERPRINT_COLUMNS = [
    "broad_area",
    "micro_area",
    "area_sqft",
    "bedrooms",
    "bathrooms",
    "base_rent_bdt",
]

normalized_fingerprint = (
    source_unique_df[NORMALIZED_FINGERPRINT_COLUMNS]
    .astype("string")
    .fillna("<MISSING>")
    .agg("||".join, axis=1)
)

group_codes, unique_group_values = pd.factorize(
    normalized_fingerprint,
    sort=True,
)

source_unique_df["duplicate_group_id"] = [
    f"DG-{code + 1:07d}"
    for code in group_codes
]

source_unique_df["normalized_group_size"] = (
    source_unique_df.groupby("duplicate_group_id")["record_id"]
    .transform("size")
)

duplicate_summary_df = (
    source_unique_df.groupby(
        [
            "duplicate_group_id",
            *NORMALIZED_FINGERPRINT_COLUMNS,
        ],
        dropna=False,
    )
    .size()
    .reset_index(name="group_size")
    .sort_values("group_size", ascending=False)
)

DUPLICATE_SUMMARY_PATH = INTERIM_DIR / "duplicate_summary.csv"

duplicate_summary_df.to_csv(
    DUPLICATE_SUMMARY_PATH,
    index=False,
)

print(
    "Normalized duplicate groups:",
    source_unique_df["duplicate_group_id"].nunique(),
)

print(
    "Rows in normalized groups larger than one:",
    int((source_unique_df["normalized_group_size"] > 1).sum()),
)

display(duplicate_summary_df.head(30))

Normalized duplicate groups: 13901
Rows in normalized groups larger than one: 2165


,duplicate_group_id,broad_area,micro_area,area_sqft,bedrooms,bathrooms,base_rent_bdt,group_size
8570,DG-0008571,Mirpur,Section 11,650.0,2.0,2.0,11000.0,17
8568,DG-0008569,Mirpur,Section 11,650.0,2.0,2.0,10000.0,16
8572,DG-0008573,Mirpur,Section 11,650.0,2.0,2.0,12000.0,16
9103,DG-0009104,Mirpur,Section 1,650.0,2.0,2.0,12000.0,11
8007,DG-0008008,Mirpur,Pallabi,650.0,2.0,2.0,12000.0,11
9118,DG-0009119,Mirpur,Section 1,700.0,2.0,2.0,13000.0,10
8003,DG-0008004,Mirpur,Pallabi,650.0,2.0,2.0,10000.0,10
7442,DG-0007443,Mirpur,Mirpur DOHS,1100.0,2.0,2.0,25000.0,9
8033,DG-0008034,Mirpur,Pallabi,800.0,2.0,2.0,12000.0,9
7537,DG-0007538,Mirpur,Mirpur DOHS,2200.0,4.0,4.0,40000.0,9


## Invalid Values and Outliers

Records with missing, malformed, or non-positive required fields are
quarantined. Plausible extreme rentals are retained and marked for analysis.

In [37]:
def invalid_reasons(row):
    reasons = []

    if pd.isna(row["location_raw"]) or not str(row["location_raw"]).strip():
        reasons.append("missing_location")

    if row["broad_area"] == "Unknown":
        reasons.append("unresolved_broad_area")

    if pd.isna(row["area_sqft"]):
        reasons.append("area_parse_failure")
    elif row["area_sqft"] <= 0:
        reasons.append("non_positive_area")

    if pd.isna(row["bedrooms"]):
        reasons.append("bedroom_parse_failure")
    elif row["bedrooms"] <= 0:
        reasons.append("non_positive_bedrooms")

    if pd.isna(row["bathrooms"]):
        reasons.append("bathroom_parse_failure")
    elif row["bathrooms"] <= 0:
        reasons.append("non_positive_bathrooms")

    if pd.isna(row["base_rent_bdt"]):
        reasons.append("rent_parse_failure")
    elif row["base_rent_bdt"] <= 0:
        reasons.append("non_positive_rent")

    return ";".join(reasons)


source_unique_df["invalid_reasons"] = source_unique_df.apply(
    invalid_reasons,
    axis=1,
)

source_unique_df["is_invalid"] = (
    source_unique_df["invalid_reasons"].str.len() > 0
)

quarantined_records_df = source_unique_df.loc[
    source_unique_df["is_invalid"]
].copy()

QUARANTINE_PATH = INTERIM_DIR / "quarantined_records.csv"

quarantined_records_df.to_csv(
    QUARANTINE_PATH,
    index=False,
)

valid_df = source_unique_df.loc[
    ~source_unique_df["is_invalid"]
].copy()

print("Quarantined invalid rows:", len(quarantined_records_df))
print("Remaining valid rows:", len(valid_df))

if len(quarantined_records_df) > 0:
    display(
        quarantined_records_df[
            [
                "record_id",
                "Location",
                "Area",
                "Bed",
                "Bath",
                "Price",
                "invalid_reasons",
            ]
        ].head(30)
    )

Quarantined invalid rows: 11
Remaining valid rows: 15248


,record_id,Location,Area,Bed,Bath,Price,invalid_reasons
1195,DN-001196,Dhaka,700 sqft,2,1,10.5 Thousand,unresolved_broad_area
3213,DN-003214,Dhaka,"1,050 sqft",3,2,20 Thousand,unresolved_broad_area
9376,DN-009377,Dhaka,"1,100 sqft",3,3,18 Thousand,unresolved_broad_area
11494,DN-011495,Dhaka,"1,150 sqft",3,2,17.5 Thousand,unresolved_broad_area
16611,DN-016612,Dhaka,500 sqft,2,1,10 Thousand,unresolved_broad_area
16613,DN-016614,Dhaka,500 sqft,2,1,11 Thousand,unresolved_broad_area
16742,DN-016743,Dhaka,"1,486 sqft",3,3,20 Thousand,unresolved_broad_area
18280,DN-018281,Dhaka,"1,050 sqft",3,2,15.5 Thousand,unresolved_broad_area
19001,DN-019002,Dhaka,"1,650 sqft",3,4,36 Thousand,unresolved_broad_area
22720,DN-022721,Dhaka,"1,100 sqft",3,3,20 Thousand,unresolved_broad_area


In [38]:
valid_df["rent_per_sqft_analysis_only"] = (
    valid_df["base_rent_bdt"]
    / valid_df["area_sqft"]
)

quantile_limits = {
    "area_low": float(valid_df["area_sqft"].quantile(0.005)),
    "area_high": float(valid_df["area_sqft"].quantile(0.995)),
    "rent_low": float(valid_df["base_rent_bdt"].quantile(0.005)),
    "rent_high": float(valid_df["base_rent_bdt"].quantile(0.995)),
    "rpsf_low": float(
        valid_df["rent_per_sqft_analysis_only"].quantile(0.005)
    ),
    "rpsf_high": float(
        valid_df["rent_per_sqft_analysis_only"].quantile(0.995)
    ),
}

display(pd.DataFrame(
    quantile_limits.items(),
    columns=["threshold", "value"],
))

,threshold,value
0,area_low,400.000000
1,area_high,3200.000000
2,rent_low,8000.000000
3,rent_high,165765.000000
4,rpsf_low,10.000000
5,rpsf_high,61.636538


In [39]:
def classify_outlier(row):
    reasons = []

    if row["area_sqft"] < quantile_limits["area_low"]:
        reasons.append("unusually_small_area")

    if row["area_sqft"] > quantile_limits["area_high"]:
        reasons.append("unusually_large_area")

    if row["base_rent_bdt"] < quantile_limits["rent_low"]:
        reasons.append("unusually_low_rent")

    if row["base_rent_bdt"] > quantile_limits["rent_high"]:
        reasons.append("unusually_high_rent")

    if (
        row["rent_per_sqft_analysis_only"]
        < quantile_limits["rpsf_low"]
    ):
        reasons.append("unusually_low_rent_per_sqft")

    if (
        row["rent_per_sqft_analysis_only"]
        > quantile_limits["rpsf_high"]
    ):
        reasons.append("unusually_high_rent_per_sqft")

    if row["bedrooms"] >= 10 or row["bathrooms"] >= 10:
        reasons.append("possible_large_or_whole_building_listing")

    if not reasons:
        return pd.Series({
            "outlier_flag": False,
            "outlier_decision": "retained_regular",
            "outlier_reasons": "",
        })

    if (
        row["area_sqft"] > quantile_limits["area_high"]
        or row["bedrooms"] >= 10
        or row["bathrooms"] >= 10
    ):
        decision = "possible_whole_building_listing_retained"
    elif row["base_rent_bdt"] > quantile_limits["rent_high"]:
        decision = "valid_high_value_listing_retained"
    else:
        decision = "suspicious_but_retained"

    return pd.Series({
        "outlier_flag": True,
        "outlier_decision": decision,
        "outlier_reasons": ";".join(reasons),
    })


outlier_results = valid_df.apply(
    classify_outlier,
    axis=1,
)

valid_df = pd.concat(
    [
        valid_df,
        outlier_results,
    ],
    axis=1,
)

OUTLIER_DECISIONS_PATH = INTERIM_DIR / "outlier_decisions.csv"

valid_df[
    [
        "record_id",
        "location_raw",
        "broad_area",
        "micro_area",
        "area_sqft",
        "bedrooms",
        "bathrooms",
        "base_rent_bdt",
        "rent_per_sqft_analysis_only",
        "outlier_flag",
        "outlier_decision",
        "outlier_reasons",
    ]
].to_csv(
    OUTLIER_DECISIONS_PATH,
    index=False,
)

print("Outlier decisions:")
display(
    valid_df["outlier_decision"]
    .value_counts()
    .rename_axis("decision")
    .reset_index(name="record_count")
)

display(
    valid_df.loc[
        valid_df["outlier_flag"],
        [
            "record_id",
            "broad_area",
            "micro_area",
            "area_sqft",
            "bedrooms",
            "bathrooms",
            "base_rent_bdt",
            "outlier_decision",
            "outlier_reasons",
        ],
    ].head(30)
)

Outlier decisions:


,decision,record_count
0,retained_regular,15031
1,suspicious_but_retained,118
2,possible_whole_building_listing_retained,74
3,valid_high_value_listing_retained,25


,record_id,broad_area,micro_area,area_sqft,bedrooms,bathrooms,base_rent_bdt,outlier_decision,outlier_reasons
30,DN-000031,Baridhara,Baridhara,1524.0,3.0,3.0,100000.0,suspicious_but_retained,unusually_high_rent_per_sqft
33,DN-000034,Gulshan,Gulshan 1,3500.0,4.0,4.0,200000.0,possible_whole_building_listing_retained,unusually_large_area;unusually_high_rent
66,DN-000067,Gulshan,Gulshan 1,3680.0,3.0,4.0,200000.0,possible_whole_building_listing_retained,unusually_large_area;unusually_high_rent
81,DN-000082,Banani,Banani,3245.0,4.0,4.0,110000.0,possible_whole_building_listing_retained,unusually_large_area
112,DN-000113,Baridhara,Block K,3000.0,3.0,4.0,240000.0,valid_high_value_listing_retained,unusually_high_rent;unusually_high_rent_per_sqft
131,DN-000132,Bashundhara R/A,Block D,1100.0,2.0,2.0,140000.0,suspicious_but_retained,unusually_high_rent_per_sqft
154,DN-000155,Baridhara,Baridhara,4200.0,4.0,4.0,400000.0,possible_whole_building_listing_retained,unusually_large_area;unusually_high_rent;unusu...
181,DN-000182,Gulshan,Gulshan 2,2800.0,4.0,4.0,250000.0,valid_high_value_listing_retained,unusually_high_rent;unusually_high_rent_per_sqft
192,DN-000193,Mirpur,Middle Paikpara,1600.0,3.0,3.0,15000.0,suspicious_but_retained,unusually_low_rent_per_sqft
194,DN-000195,Adabor,Baitul Aman Housing Society,400.0,1.0,1.0,5000.0,suspicious_but_retained,unusually_low_rent


## Micro-Area Support

Micro-areas with fewer than 30 valid historical rows are grouped under an
`OTHER_<BROAD_AREA>` category for model training.

In [40]:
micro_support_df = (
    valid_df.groupby(
        ["broad_area", "micro_area"],
        dropna=False,
    )
    .size()
    .reset_index(name="micro_area_support_count")
    .sort_values(
        "micro_area_support_count",
        ascending=False,
    )
)

valid_df = valid_df.merge(
    micro_support_df,
    on=["broad_area", "micro_area"],
    how="left",
    validate="many_to_one",
)


def category_slug(value):
    value = normalize_unicode_text(value).upper()
    value = re.sub(r"[^A-Z0-9]+", "_", value)
    value = re.sub(r"_+", "_", value).strip("_")

    return value or "UNKNOWN"


valid_df["model_micro_area"] = np.where(
    valid_df["micro_area_support_count"]
    >= MICRO_AREA_SUPPORT_THRESHOLD,
    valid_df["micro_area"],
    valid_df["broad_area"].apply(
        lambda area: f"OTHER_{category_slug(area)}"
    ),
)

valid_df["micro_area_support_level"] = pd.cut(
    valid_df["micro_area_support_count"],
    bins=[-np.inf, 29, 99, np.inf],
    labels=["low", "normal", "high"],
)

MICRO_SUPPORT_PATH = (
    PROCESSED_DIR / "location_support_counts.csv"
)

micro_support_df.to_csv(
    MICRO_SUPPORT_PATH,
    index=False,
)

print("Micro-areas:", len(micro_support_df))

print(
    "Supported micro-areas:",
    int(
        (
            micro_support_df["micro_area_support_count"]
            >= MICRO_AREA_SUPPORT_THRESHOLD
        ).sum()
    ),
)

print(
    "Rare micro-areas:",
    int(
        (
            micro_support_df["micro_area_support_count"]
            < MICRO_AREA_SUPPORT_THRESHOLD
        ).sum()
    ),
)

display(micro_support_df.head(30))

Micro-areas: 390
Supported micro-areas: 146
Rare micro-areas: 244


,broad_area,micro_area,micro_area_support_count
263,Mirpur,Section 12,453
257,Mirpur,Pallabi,352
260,Mirpur,Section 1,298
132,Dhanmondi,Dhanmondi,290
291,Mohammadpur,Mohammadpur,283
254,Mirpur,Mirpur DOHS,248
261,Mirpur,Section 10,247
253,Mirpur,Mirpur,236
258,Mirpur,Pirerbag,231
279,Mohammadpur,Bochila,231


## Final Processed Dataset

The final model dataset contains only stable identifiers, normalized
locations, numeric property fields, the regression target, and the duplicate
group identifier.

In [41]:
FINAL_COLUMNS = [
    "record_id",
    "location_raw",
    "broad_area",
    "micro_area",
    "model_micro_area",
    "micro_area_support_count",
    "area_sqft",
    "bedrooms",
    "bathrooms",
    "base_rent_bdt",
    "duplicate_group_id",
]

processed_df = (
    valid_df[FINAL_COLUMNS]
    .copy()
    .sort_values("record_id")
    .reset_index(drop=True)
)

integer_columns = [
    "micro_area_support_count",
    "area_sqft",
    "bedrooms",
    "bathrooms",
    "base_rent_bdt",
]

for column in integer_columns:
    processed_df[column] = (
        processed_df[column]
        .round()
        .astype("int64")
    )

PROCESSED_DATA_PATH = (
    PROCESSED_DIR / "dhakanest_rent_training_v1.csv"
)

processed_df.to_csv(
    PROCESSED_DATA_PATH,
    index=False,
)

print("Final processed shape:", processed_df.shape)
print("Saved:", PROCESSED_DATA_PATH)

display(processed_df.head())

Final processed shape: (15248, 11)
Saved: /content/drive/MyDrive/DhakaNest_ML/data/processed/dhakanest_rent_training_v1.csv


,record_id,location_raw,broad_area,micro_area,model_micro_area,micro_area_support_count,area_sqft,bedrooms,bathrooms,base_rent_bdt,duplicate_group_id
0,DN-000001,"Block H, Bashundhara R-A, Dhaka",Bashundhara R/A,Block H,Block H,35,1600,3,3,20000,DG-0003007
1,DN-000002,"Farmgate, Tejgaon, Dhaka",Tejgaon,Farmgate,OTHER_TEJGAON,20,900,2,2,20000,DG-0012437
2,DN-000003,"Block B, Nobodoy Housing Society, Mohammadpur,...",Mohammadpur,Nobodoy Housing Society,Nobodoy Housing Society,129,1250,3,3,18000,DG-0010784
3,DN-000004,"Gulshan 1, Gulshan, Dhaka",Gulshan,Gulshan 1,Gulshan 1,82,2200,3,4,75000,DG-0004425
4,DN-000005,"Baridhara, Dhaka",Baridhara,Baridhara,Baridhara,73,2200,3,3,75000,DG-0002008


In [42]:
sensitivity_deduplicated_df = (
    processed_df
    .drop_duplicates(
        subset=[
            "broad_area",
            "micro_area",
            "area_sqft",
            "bedrooms",
            "bathrooms",
            "base_rent_bdt",
        ],
        keep="first",
    )
    .copy()
)

SENSITIVITY_DATA_PATH = (
    PROCESSED_DIR
    / "dhakanest_rent_training_v1_normalized_deduplicated.csv"
)

sensitivity_deduplicated_df.to_csv(
    SENSITIVITY_DATA_PATH,
    index=False,
)

print("Official retained dataset:", len(processed_df))
print("Sensitivity deduplicated dataset:", len(sensitivity_deduplicated_df))
print(
    "Difference:",
    len(processed_df) - len(sensitivity_deduplicated_df),
)

Official retained dataset: 15248
Sensitivity deduplicated dataset: 13890
Difference: 1358


In [43]:
REQUIRED_MODEL_COLUMNS = [
    "broad_area",
    "model_micro_area",
    "area_sqft",
    "bedrooms",
    "bathrooms",
    "base_rent_bdt",
]

assert processed_df["record_id"].is_unique, (
    "record_id is not unique"
)

assert processed_df["record_id"].notna().all(), (
    "record_id contains missing values"
)

assert processed_df[REQUIRED_MODEL_COLUMNS].notna().all().all(), (
    "Required model columns contain missing values"
)

assert (processed_df["area_sqft"] > 0).all()
assert (processed_df["bedrooms"] > 0).all()
assert (processed_df["bathrooms"] > 0).all()
assert (processed_df["base_rent_bdt"] > 0).all()

assert (
    processed_df["micro_area_support_count"] > 0
).all()

print("=" * 60)
print("PROCESSED DATASET VALIDATION PASSED")
print("=" * 60)
print("Rows:", len(processed_df))
print("Broad areas:", processed_df["broad_area"].nunique())
print("Micro-areas:", processed_df["micro_area"].nunique())
print(
    "Model micro-area categories:",
    processed_df["model_micro_area"].nunique(),
)

PROCESSED DATASET VALIDATION PASSED
Rows: 15248
Broad areas: 68
Micro-areas: 353
Model micro-area categories: 198


## Frozen Data Splitting

Duplicate groups remain entirely within one split. Candidate group-based
splits are evaluated for row-count, rent-band, and broad-area balance.

In [44]:
split_work_df = processed_df.copy()

split_work_df["rent_band"] = pd.cut(
    split_work_df["base_rent_bdt"],
    bins=[
        -np.inf,
        15_000,
        30_000,
        60_000,
        np.inf,
    ],
    labels=[
        "below_15000",
        "15000_to_30000",
        "30001_to_60000",
        "above_60000",
    ],
)

broad_area_counts = (
    split_work_df["broad_area"]
    .value_counts()
)

split_work_df["split_broad_area"] = np.where(
    split_work_df["broad_area"].map(broad_area_counts) >= 50,
    split_work_df["broad_area"],
    "OTHER_RARE_BROAD_AREA",
)

display(
    split_work_df["rent_band"]
    .value_counts()
    .sort_index()
    .rename_axis("rent_band")
    .reset_index(name="record_count")
)

display(
    split_work_df["split_broad_area"]
    .value_counts()
    .head(30)
    .rename_axis("broad_area")
    .reset_index(name="record_count")
)

,rent_band,record_count
0,below_15000,6037
1,15000_to_30000,7090
2,30001_to_60000,1608
3,above_60000,513


,broad_area,record_count
0,Mirpur,3682
1,Mohammadpur,1739
2,Uttara,1207
3,Badda,882
4,Bashundhara R/A,828
5,OTHER_RARE_BROAD_AREA,569
6,Banasree,537
7,Dakshin Khan,438
8,Dhanmondi,416
9,Khilgaon,318


In [45]:
TARGET_RATIOS = {
    "train": 0.70,
    "validation": 0.15,
    "test": 0.15,
}


def normalized_distribution(dataframe, column):
    return (
        dataframe[column]
        .value_counts(normalize=True, dropna=False)
    )


def distribution_distance(
    dataframe,
    column,
    reference_distribution,
):
    current_distribution = normalized_distribution(
        dataframe,
        column,
    )

    categories = (
        reference_distribution.index
        .union(current_distribution.index)
    )

    reference_aligned = reference_distribution.reindex(
        categories,
        fill_value=0,
    )

    current_aligned = current_distribution.reindex(
        categories,
        fill_value=0,
    )

    return float(
        np.abs(reference_aligned - current_aligned).sum()
    )


overall_rent_distribution = normalized_distribution(
    split_work_df,
    "rent_band",
)

overall_broad_distribution = normalized_distribution(
    split_work_df,
    "split_broad_area",
)


def evaluate_split_candidate(
    train_candidate,
    validation_candidate,
    test_candidate,
):
    total_rows = len(split_work_df)

    actual_ratios = {
        "train": len(train_candidate) / total_rows,
        "validation": len(validation_candidate) / total_rows,
        "test": len(test_candidate) / total_rows,
    }

    size_error = sum(
        abs(actual_ratios[name] - TARGET_RATIOS[name])
        for name in TARGET_RATIOS
    )

    rent_error = sum(
        distribution_distance(
            candidate,
            "rent_band",
            overall_rent_distribution,
        )
        for candidate in [
            train_candidate,
            validation_candidate,
            test_candidate,
        ]
    )

    broad_error = sum(
        distribution_distance(
            candidate,
            "split_broad_area",
            overall_broad_distribution,
        )
        for candidate in [
            train_candidate,
            validation_candidate,
            test_candidate,
        ]
    )

    total_score = (
        10.0 * size_error
        + 2.0 * rent_error
        + 1.0 * broad_error
    )

    return {
        "score": total_score,
        "size_error": size_error,
        "rent_error": rent_error,
        "broad_error": broad_error,
        **actual_ratios,
    }

In [46]:
best_split = None
best_evaluation = None

outer_splitter = GroupShuffleSplit(
    n_splits=100,
    test_size=0.30,
    random_state=RANDOM_STATE,
)

groups = split_work_df["duplicate_group_id"]

for outer_index, (train_indices, temporary_indices) in enumerate(
    outer_splitter.split(
        split_work_df,
        groups=groups,
    )
):
    train_candidate = split_work_df.iloc[
        train_indices
    ].copy()

    temporary_candidate = split_work_df.iloc[
        temporary_indices
    ].copy()

    inner_splitter = GroupShuffleSplit(
        n_splits=10,
        test_size=0.50,
        random_state=RANDOM_STATE + outer_index,
    )

    temporary_groups = temporary_candidate[
        "duplicate_group_id"
    ]

    for validation_relative, test_relative in inner_splitter.split(
        temporary_candidate,
        groups=temporary_groups,
    ):
        validation_candidate = temporary_candidate.iloc[
            validation_relative
        ].copy()

        test_candidate = temporary_candidate.iloc[
            test_relative
        ].copy()

        evaluation = evaluate_split_candidate(
            train_candidate,
            validation_candidate,
            test_candidate,
        )

        if (
            best_evaluation is None
            or evaluation["score"] < best_evaluation["score"]
        ):
            best_evaluation = evaluation

            best_split = (
                train_candidate.copy(),
                validation_candidate.copy(),
                test_candidate.copy(),
            )

train_df, validation_df, test_df = best_split

print("Best split evaluation:")
display(pd.DataFrame([best_evaluation]))

Best split evaluation:


,score,size_error,rent_error,broad_error,train,validation,test
0,0.218511,0.000735,0.023101,0.164964,0.699633,0.150118,0.150249


In [47]:
train_groups = set(train_df["duplicate_group_id"])
validation_groups = set(validation_df["duplicate_group_id"])
test_groups = set(test_df["duplicate_group_id"])

assert train_groups.isdisjoint(validation_groups), (
    "Duplicate-group leakage between train and validation"
)

assert train_groups.isdisjoint(test_groups), (
    "Duplicate-group leakage between train and test"
)

assert validation_groups.isdisjoint(test_groups), (
    "Duplicate-group leakage between validation and test"
)

train_ids = set(train_df["record_id"])
validation_ids = set(validation_df["record_id"])
test_ids = set(test_df["record_id"])

assert train_ids.isdisjoint(validation_ids)
assert train_ids.isdisjoint(test_ids)
assert validation_ids.isdisjoint(test_ids)

assert (
    len(train_df)
    + len(validation_df)
    + len(test_df)
    == len(processed_df)
)

print("No record or duplicate-group leakage detected.")

No record or duplicate-group leakage detected.


In [48]:
SAVE_COLUMNS = FINAL_COLUMNS

train_df = (
    train_df[SAVE_COLUMNS]
    .sort_values("record_id")
    .reset_index(drop=True)
)

validation_df = (
    validation_df[SAVE_COLUMNS]
    .sort_values("record_id")
    .reset_index(drop=True)
)

test_df = (
    test_df[SAVE_COLUMNS]
    .sort_values("record_id")
    .reset_index(drop=True)
)

TRAIN_PATH = SPLITS_DIR / "train.csv"
VALIDATION_PATH = SPLITS_DIR / "validation.csv"
TEST_PATH = SPLITS_DIR / "test.csv"

train_df.to_csv(TRAIN_PATH, index=False)
validation_df.to_csv(VALIDATION_PATH, index=False)
test_df.to_csv(TEST_PATH, index=False)

print("Saved:", TRAIN_PATH)
print("Saved:", VALIDATION_PATH)
print("Saved:", TEST_PATH)

Saved: /content/drive/MyDrive/DhakaNest_ML/data/splits/train.csv
Saved: /content/drive/MyDrive/DhakaNest_ML/data/splits/validation.csv
Saved: /content/drive/MyDrive/DhakaNest_ML/data/splits/test.csv


In [49]:
manifest_parts = []

for split_name, dataframe in [
    ("train", train_df),
    ("validation", validation_df),
    ("test", test_df),
]:
    part = dataframe[
        [
            "record_id",
            "duplicate_group_id",
            "broad_area",
            "micro_area",
            "base_rent_bdt",
        ]
    ].copy()

    part["split"] = split_name

    part["rent_band"] = pd.cut(
        part["base_rent_bdt"],
        bins=[
            -np.inf,
            15_000,
            30_000,
            60_000,
            np.inf,
        ],
        labels=[
            "below_15000",
            "15000_to_30000",
            "30001_to_60000",
            "above_60000",
        ],
    )

    manifest_parts.append(part)

split_manifest_df = pd.concat(
    manifest_parts,
    ignore_index=True,
)

SPLIT_MANIFEST_PATH = SPLITS_DIR / "split_manifest.csv"

split_manifest_df.to_csv(
    SPLIT_MANIFEST_PATH,
    index=False,
)

print("Saved:", SPLIT_MANIFEST_PATH)
display(split_manifest_df.head())

Saved: /content/drive/MyDrive/DhakaNest_ML/data/splits/split_manifest.csv


,record_id,duplicate_group_id,broad_area,micro_area,base_rent_bdt,split,rent_band
0,DN-000001,DG-0003007,Bashundhara R/A,Block H,20000,train,15000_to_30000
1,DN-000002,DG-0012437,Tejgaon,Farmgate,20000,train,15000_to_30000
2,DN-000003,DG-0010784,Mohammadpur,Nobodoy Housing Society,18000,train,15000_to_30000
3,DN-000007,DG-0002015,Baridhara,Baridhara,75000,train,above_60000
4,DN-000008,DG-0010998,Mohammadpur,Pc Culture Housing,14000,train,below_15000


In [50]:
split_summary_rows = []

for split_name, dataframe in [
    ("train", train_df),
    ("validation", validation_df),
    ("test", test_df),
]:
    split_summary_rows.append({
        "split": split_name,
        "rows": len(dataframe),
        "percentage": round(
            len(dataframe) / len(processed_df) * 100,
            4,
        ),
        "duplicate_groups": dataframe[
            "duplicate_group_id"
        ].nunique(),
        "broad_areas": dataframe["broad_area"].nunique(),
        "micro_areas": dataframe["micro_area"].nunique(),
        "minimum_rent": int(
            dataframe["base_rent_bdt"].min()
        ),
        "median_rent": float(
            dataframe["base_rent_bdt"].median()
        ),
        "maximum_rent": int(
            dataframe["base_rent_bdt"].max()
        ),
    })

split_summary_df = pd.DataFrame(split_summary_rows)

SPLIT_SUMMARY_PATH = SPLITS_DIR / "split_summary.csv"

split_summary_df.to_csv(
    SPLIT_SUMMARY_PATH,
    index=False,
)

display(split_summary_df)

,split,rows,percentage,duplicate_groups,broad_areas,micro_areas,minimum_rent,median_rent,maximum_rent
0,train,10668,69.9633,9723,68,342,5000,17000.0,650000
1,validation,2289,15.0118,2083,64,264,6500,17000.0,350000
2,test,2291,15.0249,2084,61,265,6300,17000.0,350000


In [51]:
challenge_source_df = train_df.copy()

challenge_source_df["micro_area_key"] = (
    challenge_source_df["broad_area"].astype(str)
    + "||"
    + challenge_source_df["micro_area"].astype(str)
)

training_micro_counts = (
    challenge_source_df["micro_area_key"]
    .value_counts()
)

eligible_micro_areas = training_micro_counts.loc[
    training_micro_counts >= MICRO_AREA_SUPPORT_THRESHOLD
].copy()

rng = np.random.default_rng(RANDOM_STATE)

eligible_keys = eligible_micro_areas.index.to_numpy().copy()
rng.shuffle(eligible_keys)

target_challenge_rows = int(
    round(len(challenge_source_df) * 0.10)
)

selected_micro_keys = []
selected_row_count = 0

for key in eligible_keys:
    key_count = int(eligible_micro_areas.loc[key])

    if (
        selected_micro_keys
        and selected_row_count >= target_challenge_rows
    ):
        break

    selected_micro_keys.append(key)
    selected_row_count += key_count

print("Eligible supported micro-areas:", len(eligible_micro_areas))
print("Selected challenge micro-areas:", len(selected_micro_keys))
print("Approximate challenge rows:", selected_row_count)

Eligible supported micro-areas: 114
Selected challenge micro-areas: 14
Approximate challenge rows: 1119


In [52]:
challenge_mask = challenge_source_df[
    "micro_area_key"
].isin(selected_micro_keys)

unseen_challenge_df = (
    challenge_source_df.loc[challenge_mask, SAVE_COLUMNS]
    .sort_values("record_id")
    .reset_index(drop=True)
)

unseen_training_df = (
    challenge_source_df.loc[~challenge_mask, SAVE_COLUMNS]
    .sort_values("record_id")
    .reset_index(drop=True)
)

assert set(unseen_training_df["record_id"]).isdisjoint(
    set(unseen_challenge_df["record_id"])
)

train_micro_keys_after_holdout = set(
    unseen_training_df["broad_area"].astype(str)
    + "||"
    + unseen_training_df["micro_area"].astype(str)
)

assert train_micro_keys_after_holdout.isdisjoint(
    set(selected_micro_keys)
)

UNSEEN_TRAIN_PATH = (
    SPLITS_DIR / "unseen_micro_area_train.csv"
)

UNSEEN_CHALLENGE_PATH = (
    SPLITS_DIR / "unseen_micro_area_challenge.csv"
)

UNSEEN_MANIFEST_PATH = (
    SPLITS_DIR / "unseen_micro_area_manifest.csv"
)

unseen_training_df.to_csv(
    UNSEEN_TRAIN_PATH,
    index=False,
)

unseen_challenge_df.to_csv(
    UNSEEN_CHALLENGE_PATH,
    index=False,
)

unseen_manifest_df = pd.DataFrame({
    "micro_area_key": selected_micro_keys,
})

unseen_manifest_df[["broad_area", "micro_area"]] = (
    unseen_manifest_df["micro_area_key"]
    .str.split(r"\|\|", n=1, expand=True)
)

unseen_manifest_df["record_count"] = (
    unseen_manifest_df["micro_area_key"]
    .map(training_micro_counts)
)

unseen_manifest_df.to_csv(
    UNSEEN_MANIFEST_PATH,
    index=False,
)

print("Unseen-training rows:", len(unseen_training_df))
print("Unseen challenge rows:", len(unseen_challenge_df))

display(unseen_manifest_df)

Unseen-training rows: 9549
Unseen challenge rows: 1119


,micro_area_key,broad_area,micro_area,record_count
0,Mirpur||East Kazipara,Mirpur,East Kazipara,52
1,Khilgaon||Goran,Khilgaon,Goran,83
2,Mirpur||West Shewrapara,Mirpur,West Shewrapara,83
3,Mirpur||Section 12,Mirpur,Section 12,339
4,Malibagh||Gulbag,Malibagh,Gulbag,32
5,Mohammadpur||Shekhertek,Mohammadpur,Shekhertek,56
6,Mirpur||Rupnagar R/A,Mirpur,Rupnagar R/A,60
7,Hatirpool||Hatirpool,Hatirpool,Hatirpool,32
8,Tejgaon||Tejgaon,Tejgaon,Tejgaon,47
9,Bashundhara R/A||Block I,Bashundhara R/A,Block I,60


In [53]:
def calculate_sha256(file_path):
    sha256 = hashlib.sha256()

    with Path(file_path).open("rb") as file:
        for block in iter(
            lambda: file.read(1024 * 1024),
            b"",
        ):
            sha256.update(block)

    return sha256.hexdigest()


file_checksums = {
    "processed_dataset": calculate_sha256(
        PROCESSED_DATA_PATH
    ),
    "train": calculate_sha256(TRAIN_PATH),
    "validation": calculate_sha256(VALIDATION_PATH),
    "test": calculate_sha256(TEST_PATH),
    "split_manifest": calculate_sha256(
        SPLIT_MANIFEST_PATH
    ),
}

display(pd.DataFrame(
    file_checksums.items(),
    columns=["file", "sha256"],
))

,file,sha256
0,processed_dataset,451eecf5e67460d74095e1a87a4d009255c9ab73ba3e09...
1,train,41036273c98520726badbbb22903f37aa1e99591bb33d1...
2,validation,a984aaadede9d6568c339523c3f2479a6f0282debcab84...
3,test,05de67d08b0eeca17c7916324ce5a2e1cec1fe1fa060c5...
4,split_manifest,ac22ad4e0726eac45fe82d78afc3c74c2805c90f6e6c6a...


In [54]:
preparation_manifest = {
    "project_name": "DhakaNest",
    "pipeline_stage": "data_preparation_v1",
    "created_at_utc": datetime.now(
        timezone.utc
    ).isoformat(),
    "python_version": platform.python_version(),
    "pandas_version": pd.__version__,
    "numpy_version": np.__version__,
    "scikit_learn_version": sklearn.__version__,
    "random_state": RANDOM_STATE,
    "micro_area_support_threshold": (
        MICRO_AREA_SUPPORT_THRESHOLD
    ),
    "raw_rows": int(len(raw_df)),
    "exact_duplicate_rows_removed": int(
        working_df["is_exact_source_duplicate"].sum()
    ),
    "invalid_rows_quarantined": int(
        len(quarantined_records_df)
    ),
    "final_processed_rows": int(len(processed_df)),
    "train_rows": int(len(train_df)),
    "validation_rows": int(len(validation_df)),
    "test_rows": int(len(test_df)),
    "unseen_challenge_rows": int(
        len(unseen_challenge_df)
    ),
    "feature_columns": [
        "broad_area",
        "model_micro_area",
        "area_sqft",
        "bedrooms",
        "bathrooms",
    ],
    "target_column": "base_rent_bdt",
    "checksums": file_checksums,
}

PREPARATION_MANIFEST_PATH = (
    MANIFESTS_DIR / "data_preparation_manifest.json"
)

with PREPARATION_MANIFEST_PATH.open(
    "w",
    encoding="utf-8",
) as file:
    json.dump(
        preparation_manifest,
        file,
        indent=2,
    )

print("Saved:", PREPARATION_MANIFEST_PATH)

Saved: /content/drive/MyDrive/DhakaNest_ML/manifests/data_preparation_manifest.json


In [55]:
cleaning_report = f"""# DhakaNest Data Cleaning Report

## Dataset

- Original rows: {len(raw_df):,}
- Original columns: {raw_df.shape[1]}
- Exact source duplicates removed: {int(working_df["is_exact_source_duplicate"].sum()):,}
- Invalid records quarantined: {len(quarantined_records_df):,}
- Final processed records: {len(processed_df):,}

## Parsing

The following transformations were applied:

- `Area` to `area_sqft`
- `Bed` to `bedrooms`
- `Bath` to `bathrooms`
- `Price` to `base_rent_bdt`
- `Location` to `location_raw`

Malformed values were converted to missing values and quarantined.
They were not silently converted to zero.

## Location Normalization

- Broad areas: {processed_df["broad_area"].nunique():,}
- Raw micro-areas: {processed_df["micro_area"].nunique():,}
- Model micro-area categories: {processed_df["model_micro_area"].nunique():,}
- Minimum support threshold: {MICRO_AREA_SUPPORT_THRESHOLD}

Micro-areas below the threshold were mapped to an
`OTHER_<BROAD_AREA>` model category.

## Duplicate Policy

Exact source duplicates were removed.

Repeated normalized combinations were retained because separate rental
properties may share identical location, area, bedroom, bathroom, and rent
values. These records were assigned a shared `duplicate_group_id` and kept
within the same dataset split.

## Outlier Policy

Records with missing or non-positive required values were quarantined.

Plausible high-value, unusually large, unusually small, and whole-building
records were retained with documented outlier labels.

Rent per square foot was used only for data analysis and was not included as
a model input.

## Frozen Splits

- Training rows: {len(train_df):,}
- Validation rows: {len(validation_df):,}
- Test rows: {len(test_df):,}
- Random state: {RANDOM_STATE}

The test dataset must remain untouched until the winning algorithm has been
selected using validation performance.
"""

CLEANING_REPORT_PATH = (
    REPORTS_DIR / "data_cleaning_report.md"
)

CLEANING_REPORT_PATH.write_text(
    cleaning_report,
    encoding="utf-8",
)

print("Saved:", CLEANING_REPORT_PATH)

Saved: /content/drive/MyDrive/DhakaNest_ML/reports/data_cleaning_report.md


In [56]:
location_report = f"""# DhakaNest Location Normalization Report

## Purpose

The original location strings were converted into broad-area, micro-area,
and optional sub-area components.

## Results

- Unique raw locations: {raw_df["Location"].nunique(dropna=True):,}
- Normalized broad areas: {processed_df["broad_area"].nunique():,}
- Normalized micro-areas: {processed_df["micro_area"].nunique():,}
- Model micro-area categories: {processed_df["model_micro_area"].nunique():,}
- Support threshold: {MICRO_AREA_SUPPORT_THRESHOLD}
- Unresolved/conflicting reference rows before manual review:
  {len(unresolved_locations_df):,}

## Example

Raw location:

`Block C, Section 10, Mirpur, Dhaka`

Normalized values:

- broad area: `Mirpur`
- micro-area: `Section 10`
- sub-area detail: `Block C`

## Fallback Preparation

A micro-area with fewer than {MICRO_AREA_SUPPORT_THRESHOLD} valid records is
mapped to `OTHER_<BROAD_AREA>` for the primary model. The original
micro-area remains stored for reporting and future normalization.
"""

LOCATION_REPORT_PATH = (
    REPORTS_DIR / "location_normalization_report.md"
)

LOCATION_REPORT_PATH.write_text(
    location_report,
    encoding="utf-8",
)

print("Saved:", LOCATION_REPORT_PATH)

Saved: /content/drive/MyDrive/DhakaNest_ML/reports/location_normalization_report.md


In [57]:
required_output_files = [
    AUDIT_DIR / "dataset_audit.md",
    AUDIT_DIR / "raw_data_profile.csv",
    INTERIM_DIR / "parsing_failures.csv",
    INTERIM_DIR / "unresolved_locations.csv",
    INTERIM_DIR / "manual_location_overrides.csv",
    INTERIM_DIR / "confirmed_duplicate_rows.csv",
    INTERIM_DIR / "duplicate_summary.csv",
    INTERIM_DIR / "quarantined_records.csv",
    INTERIM_DIR / "outlier_decisions.csv",
    PROCESSED_DIR / "location_reference.csv",
    PROCESSED_DIR / "location_aliases.csv",
    PROCESSED_DIR / "location_support_counts.csv",
    PROCESSED_DATA_PATH,
    TRAIN_PATH,
    VALIDATION_PATH,
    TEST_PATH,
    SPLIT_MANIFEST_PATH,
    SPLITS_DIR / "split_summary.csv",
    UNSEEN_TRAIN_PATH,
    UNSEEN_CHALLENGE_PATH,
    UNSEEN_MANIFEST_PATH,
    PREPARATION_MANIFEST_PATH,
    CLEANING_REPORT_PATH,
    LOCATION_REPORT_PATH,
]

missing_output_files = [
    str(path)
    for path in required_output_files
    if not path.exists()
]

if missing_output_files:
    raise FileNotFoundError(
        "Missing output files:\n"
        + "\n".join(missing_output_files)
    )

assert len(processed_df) == (
    len(train_df)
    + len(validation_df)
    + len(test_df)
)

print("=" * 70)
print("DHAKANEST DATA PREPARATION COMPLETED SUCCESSFULLY")
print("=" * 70)
print("Raw rows:", len(raw_df))
print(
    "Exact duplicates removed:",
    int(working_df["is_exact_source_duplicate"].sum()),
)
print(
    "Invalid rows quarantined:",
    len(quarantined_records_df),
)
print("Processed rows:", len(processed_df))
print("Training rows:", len(train_df))
print("Validation rows:", len(validation_df))
print("Test rows:", len(test_df))
print(
    "Unseen challenge rows:",
    len(unseen_challenge_df),
)
print("All required files exist.")

DHAKANEST DATA PREPARATION COMPLETED SUCCESSFULLY
Raw rows: 28800
Exact duplicates removed: 13541
Invalid rows quarantined: 11
Processed rows: 15248
Training rows: 10668
Validation rows: 2289
Test rows: 2291
Unseen challenge rows: 1119
All required files exist.


In [58]:
summary_for_review = {
    "raw_shape": raw_df.shape,
    "raw_unique_locations": int(
        raw_df["Location"].nunique(dropna=True)
    ),
    "parsing_failures": int(len(parsing_failures_df)),
    "unresolved_location_rows": int(
        len(unresolved_locations_df)
    ),
    "exact_duplicates_removed": int(
        working_df["is_exact_source_duplicate"].sum()
    ),
    "invalid_rows_quarantined": int(
        len(quarantined_records_df)
    ),
    "outlier_rows_retained": int(
        valid_df["outlier_flag"].sum()
    ),
    "processed_rows": int(len(processed_df)),
    "broad_areas": int(
        processed_df["broad_area"].nunique()
    ),
    "micro_areas": int(
        processed_df["micro_area"].nunique()
    ),
    "model_micro_area_categories": int(
        processed_df["model_micro_area"].nunique()
    ),
    "train_rows": int(len(train_df)),
    "validation_rows": int(len(validation_df)),
    "test_rows": int(len(test_df)),
    "unseen_challenge_rows": int(
        len(unseen_challenge_df)
    ),
}

print(json.dumps(summary_for_review, indent=2))

{
  "raw_shape": [
    28800,
    6
  ],
  "raw_unique_locations": 730,
  "parsing_failures": 0,
  "unresolved_location_rows": 1,
  "exact_duplicates_removed": 13541,
  "invalid_rows_quarantined": 11,
  "outlier_rows_retained": 217,
  "processed_rows": 15248,
  "broad_areas": 68,
  "micro_areas": 353,
  "model_micro_area_categories": 198,
  "train_rows": 10668,
  "validation_rows": 2289,
  "test_rows": 2291,
  "unseen_challenge_rows": 1119
}


# Final Data Quality Review

Before model training, unresolved locations, normalization quality, duplicate
groups, and frozen files are reviewed.

In [59]:
print("Number of unresolved reference rows:", len(unresolved_locations_df))

display(
    unresolved_locations_df[
        [
            "location_raw",
            "location_alias_key",
            "broad_area",
            "micro_area",
            "sub_area_detail",
            "normalization_confidence",
            "record_count",
        ]
    ]
)

Number of unresolved reference rows: 1


,location_raw,location_alias_key,broad_area,micro_area,sub_area_detail,normalization_confidence,record_count
294,Dhaka,dhaka,Unknown,Unknown,,low,13


In [60]:
unresolved_alias_keys = set(
    unresolved_locations_df["location_alias_key"]
)

affected_unresolved_records = working_df.loc[
    working_df["location_alias_key"].isin(unresolved_alias_keys),
    [
        "record_id",
        "location_raw",
        "broad_area",
        "micro_area",
        "sub_area_detail",
        "normalization_confidence",
        "Area",
        "Bed",
        "Bath",
        "Price",
    ],
].copy()

print(
    "Number of original records affected:",
    len(affected_unresolved_records),
)

display(affected_unresolved_records.head(50))

Number of original records affected: 13


,record_id,location_raw,broad_area,micro_area,sub_area_detail,normalization_confidence,Area,Bed,Bath,Price
1195,DN-001196,Dhaka,Unknown,Unknown,,low,700 sqft,2,1,10.5 Thousand
3213,DN-003214,Dhaka,Unknown,Unknown,,low,"1,050 sqft",3,2,20 Thousand
9376,DN-009377,Dhaka,Unknown,Unknown,,low,"1,100 sqft",3,3,18 Thousand
11494,DN-011495,Dhaka,Unknown,Unknown,,low,"1,150 sqft",3,2,17.5 Thousand
16611,DN-016612,Dhaka,Unknown,Unknown,,low,500 sqft,2,1,10 Thousand
16612,DN-016613,Dhaka,Unknown,Unknown,,low,500 sqft,2,1,10 Thousand
16613,DN-016614,Dhaka,Unknown,Unknown,,low,500 sqft,2,1,11 Thousand
16741,DN-016742,Dhaka,Unknown,Unknown,,low,700 sqft,2,1,10.5 Thousand
16742,DN-016743,Dhaka,Unknown,Unknown,,low,"1,486 sqft",3,3,20 Thousand
18280,DN-018281,Dhaka,Unknown,Unknown,,low,"1,050 sqft",3,2,15.5 Thousand


In [61]:
unresolved_raw_locations = set(
    unresolved_locations_df["location_raw"]
)

unresolved_in_processed = processed_df.loc[
    processed_df["location_raw"].isin(unresolved_raw_locations)
]

print(
    "Unresolved locations present in final processed dataset:",
    len(unresolved_in_processed),
)

display(unresolved_in_processed)

Unresolved locations present in final processed dataset: 0


,record_id,location_raw,broad_area,micro_area,model_micro_area,micro_area_support_count,area_sqft,bedrooms,bathrooms,base_rent_bdt,duplicate_group_id


In [62]:
raw_duplicate_group_summary = (
    working_df.groupby(
        [
            "Location",
            "Area",
            "Bed",
            "Bath",
            "Price",
        ],
        dropna=False,
    )
    .agg(
        group_size=("record_id", "size"),
        first_record_id=("record_id", "first"),
    )
    .reset_index()
    .sort_values(
        ["group_size", "Location"],
        ascending=[False, True],
    )
)

repeated_raw_groups = raw_duplicate_group_summary.loc[
    raw_duplicate_group_summary["group_size"] > 1
].copy()

print("Repeated raw-property groups:", len(repeated_raw_groups))
print("Largest duplicate group:", repeated_raw_groups["group_size"].max())

display(repeated_raw_groups.head(30))

Repeated raw-property groups: 4337
Largest duplicate group: 148


,Location,Area,Bed,Bath,Price,group_size,first_record_id
3488,"Block D, Section 12, Mirpur, Dhaka",650 sqft,2,2,10 Thousand,148,DN-000037
11391,"Section 12, Mirpur, Dhaka","1,600 sqft",3,3,25 Thousand,93,DN-003467
3773,"Block E, Section 12, Mirpur, Dhaka",650 sqft,2,2,10 Thousand,74,DN-002061
11398,"Section 12, Mirpur, Dhaka","1,600 sqft",3,3,30 Thousand,73,DN-002613
3777,"Block E, Section 12, Mirpur, Dhaka",650 sqft,2,2,12 Thousand,63,DN-001423
9515,"Mohammadpur, Dhaka",720 sqft,2,2,15 Thousand,61,DN-001809
2995,"Block C, Section 12, Mirpur, Dhaka",650 sqft,2,2,10 Thousand,58,DN-002959
12967,"Shahjahanpur, Dhaka",700 sqft,2,2,20 Thousand,58,DN-004669
11393,"Section 12, Mirpur, Dhaka","1,600 sqft",3,3,26 Thousand,56,DN-004216
566,"Avenue 5, Block C, Section 11, Mirpur, Dhaka",650 sqft,2,2,10 Thousand,50,DN-002619


In [63]:
duplicate_review_summary = pd.DataFrame({
    "measure": [
        "Original raw rows",
        "Rows retained after exact deduplication",
        "Exact additional copies removed",
        "Rows belonging to repeated source groups",
        "Number of repeated source groups",
        "Largest source duplicate group",
    ],
    "value": [
        len(working_df),
        len(source_unique_df),
        int(working_df["is_exact_source_duplicate"].sum()),
        len(confirmed_duplicate_rows_df),
        len(repeated_raw_groups),
        int(repeated_raw_groups["group_size"].max()),
    ],
})

display(duplicate_review_summary)

,measure,value
0,Original raw rows,28800
1,Rows retained after exact deduplication,15259
2,Exact additional copies removed,13541
3,Rows belonging to repeated source groups,17878
4,Number of repeated source groups,4337
5,Largest source duplicate group,148


In [64]:
duplicate_policy_note = """

## Duplicate-Handling Limitation

Rows identical across Location, Area, Bed, Bath, and Price were treated as
exact dataset duplicates, and one representative from each group was
retained. This reduced repetition bias and prevented identical information
from entering multiple data splits.

The source dataset did not contain listing IDs, URLs, timestamps, or landlord
identifiers. Therefore, it was not possible to determine whether every
repeated row represented the same advertisement or a separate property with
identical attributes. This is recorded as a limitation of the source data.
"""

with CLEANING_REPORT_PATH.open("a", encoding="utf-8") as file:
    file.write(duplicate_policy_note)

print("Duplicate-handling limitation added to the cleaning report.")

Duplicate-handling limitation added to the cleaning report.
